# Volatility Modelling and Regime Classification Methods

Eloy Sentana Segui, 4th Year Bsc Mathematics and Computing, UC3M

Email: esentanasegui@gmail.com

LinkedIn: https://www.linkedin.com/in/eloysentanasegui

Github: https://github.com/eloysentana

**This is the corresponding code to the "Volatility Modelling and Regime Classification Methods" PDF. The markdown on this notebook is incomplete, please refer to the PDF for full explanations.**

## Introduction

Volatility is a key measure of risk in financial markets, central to tasks like pricing derivatives, managing portfolios, and assessing market stability. Since volatility changes over time and often clusters in periods of high or low uncertainty, accurate modeling and forecasting are essential.

Models like GARCH capture these dynamics by linking today’s volatility to past shocks and variances. Yet markets also shift between different regimes that standard models may miss. To address this, regime classification methods such as Hidden Markov Models (HMM) and Markov Switching Autoregressive (MSAR) models help identify and adapt to structural changes in volatility.

Together, these approaches provide a more complete framework for understanding and anticipating risk in financial markets.

**Motivation**

The purpose of this notebook is simple: to help me learn by doing. I wanted to get hands-on with modern volatility estimation and forecasting methods, especially GARCH models and regime-switching approaches like Hidden Markov Models (HMM) and Markov Switching Autoregressive (MSAR) models. Instead of just reading about them, I wanted to actually implement the models, see how they behave with real financial data, and understand what their results really mean.

Along the way, I’ve tried not only to reproduce results but also to question them. Why this distribution and not another? Why this algorithm, this test, this assumption? Often, digging into these “whys” has taken me beyond the main task—into the theorems behind the models, the statistical tests that validate them, and the principles of mathematical modeling in finance. Those detours have been just as valuable as the direct results.

This notebook doesn’t follow strict academic standards: it doesn’t cite everything formally, nor does it present full proofs, since the goal was to explore, to test ideas, and to build my own understanding of how volatility can be modeled and forecasted. Working with real data and algorithms has shown me something important: results in practice rarely match the clean outcomes in research papers. And that gap—between theory and reality—has been one of the most interesting parts of the journey, pushing me to ask even better questions.

I started from the work of [Yunhui Wang](https://medium.com/@yuhui_w/volatility-regime-classification-with-garch-1-1-markov-models-7cb85d4d5815), but extended his article significantly. I’ve added my own implementations, experiments, and reflections, building a kind of learning journal where curiosity and questioning are just as central as coding and results.

**Disclaimer**: This notebook has not been supervised by any academic or professional in the field. It is purely a personal learning exercise and should not be taken as financial advice or a definitive guide to volatility modeling. If you have questions, suggestions, or corrections, please feel free to reach out to esentanasegui@gmail.com

In [1]:
# Packages
import pandas as pd 
import numpy as np
import datetime as dt 

from sklearn.metrics import mean_absolute_error
import scipy.stats as stats
import statsmodels.api as sm
from hmmlearn.hmm import GaussianHMM
import arch

import yfinance as yf

import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots

from matplotlib import pyplot as plt

## Data

Throughtout this practice, we will use the daily adjusted closing prices of the S&P 500 index (^GSPC) from with a 20year lookback. The data is sourced from Yahoo Finance and can be easily accessed using the `yfinance` library in Python.

However, using the raw prices is not very useful for statistical modeling, since prices are non-stationary (they have a trend), and their magnitude changes a lot (specially in long timeframes).

Instead, we take the returns for each day, defined as the percentage change from the previous day's price. However, it is more common to use log returns. And what is their difference? Here is a comparison:


### Simple Return vs Log Return

The simple return for a price series is defined as:

$$
R_t = \frac{P_t - P_{t-1}}{P_{t-1}}
$$
- $R_t$: simple return at time $t$  
- $P_t$: price at time $t$  
- $P_{t-1}$: price at time $t-1$  

The logarithmic return (log return) for a price series is defined as:

$$
r_t = \ln\left(\frac{P_t}{P_{t-1}}\right)
$$

- $r_t$: log return at time $t$



**Understanding log returns**

Although simple returns are easier to understand, it is crucial to understand how log returns work, and why I used them on this practice.


In [2]:

import plotly.graph_objects as go
ticker_obj = yf.Ticker(str('^GSPC'))
gspc_hist = ticker_obj.history(period=str('20y'), interval=str('1d'))

prices = gspc_hist['Close'].dropna()

# Fit log-normal distribution to the prices
shape, loc, scale = stats.lognorm.fit(prices, floc=0)


# Histogram of prices
hist = go.Histogram(
    x=prices,
    nbinsx=30,
    histnorm='probability density',
    name='Price Histogram',
    marker_color='skyblue',
    opacity=0.6
)


# Fitted log-normal PDF
x = np.linspace(prices.min(), prices.max(), 1000)

pdf = stats.lognorm.pdf(x, shape, loc=0, scale=scale)
pdf_line = go.Scatter(
    x=x,
    y=pdf,
    mode='lines',
    name='Best-fit Log-normal PDF',
    line=dict(color='red', width=2)
)

fig = go.Figure([hist, pdf_line])
fig.update_layout(
    title='Histogram of ^GSPC Prices with Best-fit Log-normal Distribution',
    xaxis_title='Price',
    yaxis_title='Density',
    legend=dict(itemsizing='constant')
)
fig.show()

# Print mean and variance
mean = stats.lognorm.mean(shape, loc=0, scale=scale)
variance = stats.lognorm.var(shape, loc=0, scale=scale)
print(f"Fitted log-normal mean: {mean:.2f}")
print(f"Fitted log-normal variance: {variance:.2f}")
fig.write_image("histogram_lognormal.png", width=1700, height=1200, scale=2)  # 170 mm ~ 1700 px at 300 dpi


Fitted log-normal mean: 2520.10
Fitted log-normal variance: 2156959.28


As shown above, prices usually follow a lognormal distribution (not entirely, but it's the best option we have!) per Gregory's (very well explained) [post](https://gregorygundersen.com/blog/2022/02/06/log-returns/). We know that prices cannot go lower than 0, and (in theory), they can go up to +inf. So the support of the lognormal distribution and the prices matches.
For other models (like Black-Scholes), normality of log returns is required. So it is very convenient. 

Additionally, for small daily changes i.e. returns of +-3% (as is the case almost always), simple returns and log returns are almost identical. However, on the long run, log returns are easier to operate with, since for a chain of returns r_i, the total return is given by the product of (1 + $r_i$) for simple returns, while for log returns, it is simply the sum of the individual log returns. This property simplifies calculations over long periods of time (as it is the case for stock analysis often).

In [43]:
# Data & processing

def get_return(ticker, period, interval):
    """
    Fetch price data and compute log returns.
    Returns a pandas Series of log returns.
    """
    ticker_obj = yf.Ticker(str(ticker))
    data = ticker_obj.history(period=str(period), interval=str(interval))
    rt = np.log(data['Close']).diff().dropna()

    # Simple return
    simple_rt = data.Close.pct_change().dropna()

    return rt, simple_rt, data

def plot_return(data, rt, ticker):
    """
    Plot closing price and log returns.
    """
    # Plot the closing price
    fig = px.line(data, x=data.index, y="Close", title=f"{ticker} Closing Price")
    fig.update_layout(xaxis_title="Date", yaxis_title="Price")
    fig.write_image("closing_price.png", width=1700, height=1200, scale=2)  # 170 mm ~ 1700 px at 300 dpi
    fig.show()

    # Plot the returns
    mean_val = rt.mean()
    fig = px.line(rt, x=rt.index, y=rt, title=f"{ticker} Log Return")
    fig.add_hline(y=mean_val, line_dash="dash", line_color="red", 
                  annotation_text=f"Mean: {mean_val:.4f}", 
                  annotation_position="top left")
    fig.update_layout(
        xaxis_title="Date", 
        yaxis_title="Log Return",
        legend=dict(itemsizing='constant'),
        showlegend=True
    )
    # Add mean to legend by adding an invisible trace
    fig.add_trace(
        go.Scatter(
            x=[None], y=[None],
            mode='lines',
            line=dict(color='red', dash='dash'),
            name=f"Mean: {mean_val:.4f}"
        )
    )
    fig.write_image("log_returns.png", width=1700, height=1200, scale=2)  # 170 mm ~ 1700 px at 300 dpi
    fig.show()




In [44]:
rt, simple_rt, data = get_return(ticker='^GSPC', period='20y', interval='1d')
# Get maximum return and minimum return with dates
max_rt = rt.max()
min_rt = rt.min()
max_rt_date = rt[rt == max_rt].index[0]
min_rt_date = rt[rt == min_rt].index[0]
print(f'Maximum return: {max_rt} on {max_rt_date}')
print(f'Minimum return: {min_rt} on {min_rt_date}')

plot_return(data, rt, '^GSPC')


Maximum return: 0.10957196759533883 on 2008-10-13 00:00:00-04:00
Minimum return: -0.12765219747281709 on 2020-03-16 00:00:00-04:00


We can see the price chart of the SP500, and the chart of the log returns. Its mean is very close to 0, and we can see that there are some spikes of high volatility (mostly on crisis and negative events, like COVID, 2008 crisis, etc).

To better appreciate the difference between simple returns and log returns, let's plot them:

In [ ]:
import plotly.graph_objects as go

fig = go.Figure()

# Simple returns as red crosses
fig.add_trace(go.Scatter(
    x=simple_rt.index, y=simple_rt,
    mode='markers',
    marker=dict(symbol='x', color='red', size=7, opacity=0.5),
    name='Simple Returns'
))

# Log returns as blue circles
fig.add_trace(go.Scatter(
    x=rt.index, y=rt,
    mode='markers',
    marker=dict(symbol='circle', color='blue', size=5, opacity=0.5),
    name='Log Returns'
))

fig.update_layout(
    title="Scatter Plot of Simple vs Log Returns",
    xaxis_title="Date",
    yaxis_title="Return",
    legend=dict(itemsizing='constant')
)

fig.write_image("simple_vs_log_returns.png", width=1700, height=1200, scale=2)  # 170 mm ~ 1700 px at 300 dpi
fig.show()



We can see that they are generally very close, however, we can note the difference better when the return is very high or very low:

1. When returns are very high, the log return is lower than the simple return.
2. When returns are very low (i.e. very negative), the log return is lower (because of the "exxageration")

### The distribution of the returns

We know what distribution do prices follow (lognormal). But what distribution do returns follow? We are about to see it:

In [46]:
def return_dist_stats(simple_ret, log_ret):
    
    ################################
    # Input: simple and log return series
    # output: comparative stats and comparative chart

    # Create a DataFrame for comparative statistics
    stats_df = pd.DataFrame({
        'Simple Return': [
            simple_ret.mean() * 100,
            simple_ret.std() * 100,
            simple_ret.skew(),
            simple_ret.kurtosis(),
            stats.jarque_bera(simple_ret)[0],
            stats.jarque_bera(simple_ret)[1]
        ],
        'Log Return': [
            log_ret.mean() * 100,
            log_ret.std() * 100,
            log_ret.skew(),
            log_ret.kurtosis(),
            stats.jarque_bera(log_ret)[0],
            stats.jarque_bera(log_ret)[1]
        ]
    }, index=['Mean (%)', 'Std Dev (%)', 'Skew', 'Kurtosis', 'JB Stat', 'JB p-value'])

    print('Comparative Statistics of Return Distributions')
    print('-' * 60)
    print(stats_df.round(4))
    print()

    # Normality interpretation
    for label, ret in zip(['Simple', 'Log'], [simple_ret, log_ret]):
        jb_test = stats.jarque_bera(ret)
        if jb_test[1] < 0.05:
            print(f'{label} Return: Not normal (reject H0 at 5% significance level)')
        else:
            print(f'{label} Return: Normal (fail to reject H0 at 5% significance level)')
    print()

    # Plot both distributions on the same plot
    fig = go.Figure()
    fig.add_trace(go.Histogram(x=simple_ret, name='Simple Return', opacity=0.6, nbinsx=1000))
    fig.add_trace(go.Histogram(x=log_ret, name='Log Return', opacity=0.6, nbinsx=1000))
    fig.update_layout(
        title='Simple vs Log Return Distribution',
        xaxis_title='Return',
        yaxis_title='Count',
        barmode='overlay'
    )
    fig.write_image("ret_distribution.png", width=1700, height=1200, scale=2)  # 170 mm ~ 1700 px at 300 dpi

    fig.show()

return_dist_stats(simple_rt, rt)

Comparative Statistics of Return Distributions
------------------------------------------------------------
             Simple Return  Log Return
Mean (%)            0.0415      0.0339
Std Dev (%)         1.2268      1.2292
Skew               -0.2053     -0.4755
Kurtosis           12.7441     12.9136
JB Stat         33993.5461  35057.1092
JB p-value          0.0000      0.0000

Simple Return: Not normal (reject H0 at 5% significance level)
Log Return: Not normal (reject H0 at 5% significance level)



From these results, we can interpret the following:

- The mean daily return is slightly positive, i.e. on average the returns tend to be slighly positive.
- The standard deviation is relatively high, i.e. significant day-to-day volatility.
- The negative skewness means that large negative returns are more frequent than large positive returns.
- The high kurtosis (for a normal distribution it's 3) indicates fat tails in the return distribution, meaning extreme returns (both positive and negative) occur more often than would be expected under a normal distribution.
- The Jarque-Bera test strongly rejects the null hypothesis of normality (p-value == 0), confirming that the return distribution is not normal.

But... aren't log returns supposed to make the distribution more normal? Why is the skewness even more negative when taking the log returns? This is because the simple returns already have negative skewness, and since the log returns compress the upside and stretches the downside, the skewness becomes even more negative.

So in this case, the simple returns distribution is (slightly) closer to being a normal distribution (in all aspects except the mean: std dev, skewness, kurtosis, and JB statistic value). This is acually surprising!

However, both returns have very heavy tails, very hight kurtosis, and non-zero skewness. I.e. they are not normal (as indicated by the JB test).
And this is why we will use GARCH in this practice! Because with GARCH we can use non normal, fat tailed returns!

### Hypothesis testing for mean return and distribution normality

Even tho we have already tested for normality of the log returns (and obtained that they are far from normal), let's also test it with some arch library built-in tests.

In [69]:
def test_dist(asset_return, mode, alpha = 1e-2):
    ##############################
    # Input: return and test('mean' or 'normal'), alpha in decimals
    # output: P value and test result
    
    if (mode == 'mean'):
        t_stat, p = stats.ttest_1samp(asset_return, popmean=0, alternative='two-sided')
    elif (mode == 'normal'):
        k2, p = stats.normaltest(asset_return)
    
    def test(p, alpha):
        print("p = {:g}".format(p))
        if p < alpha: 
            print("The null hypothesis can be rejected")
        else:
            print("The null hypothesis cannot be rejected")
            
    return test(p, alpha)        
    

Let's frist test for the mean return being statistically significantly different from 0.

In [70]:
# Null hypothesis for 'mean': the mean of the return series is zero
# Alternative hypothesis: the mean of the return series is not zero

test_dist(rt, 'mean')

p = 0.0503628
The null hypothesis cannot be rejected


And now for normality:

In [9]:
# Null hypothesis for 'normal': the return series is normally distributed
# Alternative hypothesis: the return series is not normally distributed

test_dist(rt, 'normal')

p = 3.68834e-258
The null hypothesis can be rejected


From the above testing, we can tell that the mean is not statistically significantly different from 0, but it suggestes that the distribution of the log returns is not normal.

## Volatility Modelling with GARCH

The [GARCH](https://www.quantstart.com/articles/Generalised-Autoregressive-Conditional-Heteroskedasticity-GARCH-p-q-Models-for-Time-Series-Analysis) (Generalized Autoregressive Conditional Heteroskedasticity) model is used to estimate and forecast the volatility of time series, such as financial returns. It is an "advanced" version of the ARCH model introduced by [Robert Engle](https://web-static.stern.nyu.edu/rengle/GARCH101.PDF) (1982)

### Introduction

The basic idea is that the variance (volatility) at time $t$ depends on past squared returns (shocks) as well as past variances. This means that volatility tends to cluster around periods of high volatility.

But what do the acronym mean?

1. Generalized: Extends the simpler ARCH model by including past variances as explanatory variables, not just past shocks.
2. Autoregressive: The current volatility depends on its own past values (as in any [AR models](https://en.wikipedia.org/wiki/Autoregressive_model)).
3. Conditional: The variance is estimated with past information (i.e., it is not constant, but conditional on history).
4. Heteroskedasticity: The variance of the error term changes over time (we will return to this later).

The general GARCH(p, q) formula is:

$$
\sigma_t^2 = \omega + \sum_{i=1}^q \alpha_i \epsilon_{t-i}^2 + \sum_{j=1}^p \beta_j \sigma_{t-j}^2
$$

Where

$$
\begin{array}{ll}
\omega & \text{constant (baseline volatility)} \\ 
\sigma_t^2 & \text{conditional variance at time } t \\ 
\alpha_i, \beta_j & \text{parameters} \\ 
\epsilon_{t-i}^2 & \text{past squared shocks (errors)} \\ 
\sigma_{t-j}^2 & \text{past variances} \\ 
\end{array}
$$

$\omega$ ensures that the variance never collapses to zero

But... what is the shock $\epsilon_{t-i}^2$?
Mathematically, returns are modeled as:

$$
r_t = \mu_t + \varepsilon_t
$$


*Note that $\epsilon_t$ and $\varepsilon_t$ are used interchangeably (they are just variations of epsilon)*

That is, for every return, we try to predict it, but there is always some error (sometimes small, sometimes large) in our guess. That error is the shock.

And how do we get the expected return $\mu_t$? The answer is: it depends. Sometimes a constant is used; other times it is modeled with an [ARMA](https://didattica.unibocconi.it/mypage/dwload.php?nomefile=Lec_3_Autoregressive_Moving_Average_(ARMA)_Models_and_their_Practical_Applications20190212115606.pdf) process. In the case of a constant, it is obtained by "training" the model on past data. In some cases it is even set to 0 (in this case, we saw earlier that the mean was not significantly different from 0), because the main focus is volatility.

The shock is modeled as:

$$
\varepsilon_t = \sigma_t z_t
$$

where

$$
\begin{array}{ll}
\sigma_t & \text{conditional volatility at time } t \ (\text{changes over time, modeled by GARCH}) \\ 
z_t & \text{a random variable with mean 0 and variance 1 (often assumed i.i.d. normal)} \\ 
\end{array}
$$

And why do we square the shock in the GARCH formula? Because volatility must always be non-negative, and we care about its magnitude, not its direction.

With this information, we can better understand the intuition behind the GARCH model: if yesterday's volatility was high, today's volatility is also likely to be high.

Luckily for us, the most widespread version of GARCH is the GARCH(1,1) model ([see here](https://onlinelibrary.wiley.com/doi/10.1002/jae.800)):

$$
\sigma_t^2 = \omega + \alpha_1 \epsilon_{t-1}^2 + \beta_1 \sigma_{t-1}^2
$$

* Here, today's volatility depends on yesterday's squared shock and yesterday's volatility.

But there is an important property of GARCH: Covariance stationaryness

For this property, we need:

$$
\sum_{i=1}^q \alpha_i + \sum_{j=1}^p \beta_j < 1
$$

that ensures that the shocks eventually “die out,” and volatility returns to its long-run mean. If 
$
\sum_{i=1}^q \alpha_i + \sum_{j=1}^p \beta_j = 1
$
it is called IGARCH (integrated), where volatility persists at any horizon.
If 
$
\sum_{i=1}^q \alpha_i + \sum_{j=1}^p \beta_j > 1
$
 volatility explodes (i.e. it increases without bound, which is not the case for financial markets)

And what about the "Heteroskedasticity"? It simply means that the variance of the shock (error) values is not constant accross time. Let's visualize this by first plotting the shock values, and then seeing their distributions: We will split the data in 4 periods, and see how the distribution of shocks changes accross time.

In [47]:
import numpy as np
import pandas as pd
from arch import arch_model
import plotly.graph_objects as go

# === 1. Fit a simple GARCH(1,1) model ===
am = arch_model(rt, vol='Garch', p=1, q=1, mean='Constant', dist='normal', rescale=False)
res = am.fit(disp="off")

# === 2. Get residuals ===
errors = res.resid.dropna()

# Split into 4 equal parts
splits = np.array_split(errors, 4)

# Collect variance + date ranges
results = []
split_boundaries = []
for i, s in enumerate(splits):
    start = s.index[0]
    end = s.index[-1]
    var = s.var()
    results.append((f"Period {i+1}", start, end, var))
    split_boundaries.append(end)  # for plotting

var_df = pd.DataFrame(results, columns=["Period", "Start", "End", "Variance"])

# === Plotly interactive plot of residuals ===
fig = go.Figure()

# Residuals line
fig.add_trace(go.Scatter(
    x=errors.index,
    y=errors.values,
    mode="lines",
    name="Model residuals",
    line=dict(color="blue")
))

# Mean line
mean_resid = errors.mean()
fig.add_hline(
    y=mean_resid,
    line=dict(color="green", dash="dot"),
    annotation_text=f"Mean = {mean_resid:.4f}",
    annotation_position="bottom right"
)

# Vertical split lines (using add_shape for datetime safety)
for boundary in split_boundaries[:-1]:
    fig.add_shape(
        type="line",
        x0=boundary,
        x1=boundary,
        y0=errors.min(),
        y1=errors.max(),
        line=dict(color="red", dash="dash"),
        xref="x",
        yref="y"
    )

# Layout
fig.update_layout(
    title="Model Residuals with Period Splits and Mean",
    xaxis_title="Date",
    yaxis_title="Residuals",
    template="plotly_white",
    legend=dict(yanchor="top", y=0.99, xanchor="left", x=0.01)
)
fig.write_image("period_splits.png", width=1700, height=1200, scale=2)  # 170 mm ~ 1700 px at 300 dpi

fig.show()


c:\Users\esent\Desktop\bitpredict\venv\Lib\site-packages\arch\univariate\base.py:768: ConvergenceWarning:

The optimizer returned code 4. The message is:
Inequality constraints incompatible
See scipy.optimize.fmin_slsqp for code meaning.




Here we can see the residuals (i.e. the shocks). if we remember, a return is defined as $r_t = \mu_t + \varepsilon_t$ 

where $\mu_t$ is the expected return (which we set as a constant, found by the model training), and $\varepsilon_t$ is the shock (error) at time t. So this graph plots the "surprise" we got each day I.e. actual return - expected return

However, since the variance is hard to see like this, let's get it numerically:

In [11]:
print(var_df)


     Period                     Start                       End  Variance
0  Period 1 2005-09-26 00:00:00-04:00 2010-09-23 00:00:00-04:00  0.000247
1  Period 2 2010-09-24 00:00:00-04:00 2015-09-22 00:00:00-04:00  0.000093
2  Period 3 2015-09-23 00:00:00-04:00 2020-09-18 00:00:00-04:00  0.000147
3  Period 4 2020-09-21 00:00:00-04:00 2025-09-22 00:00:00-04:00  0.000117


In [48]:
import plotly.graph_objects as go
from plotly.subplots import make_subplots
import numpy as np
from scipy.stats import gaussian_kde

# === Create 2x2 subplot grid ===
fig = make_subplots(rows=2, cols=2, subplot_titles=[
    f"{var_df['Period'][i]} ({var_df['Start'][i].date()} to {var_df['End'][i].date()})<br>Var = {splits[i].var():.6f}"
    for i in range(4)
])

# === Loop through splits ===
for i, s in enumerate(splits):
    row = i // 2 + 1
    col = i % 2 + 1

    mean_val = s.mean()
    var_val = s.var()

    # Histogram (normalized to density for KDE overlay)
    fig.add_trace(
        go.Histogram(
            x=s.values,
            nbinsx=30,
            histnorm="probability density",
            marker_color="skyblue",
            showlegend=False
        ),
        row=row, col=col
    )

    # KDE curve (blue line)
    kde = gaussian_kde(s.values)
    x_range = np.linspace(s.min(), s.max(), 200)
    fig.add_trace(
        go.Scatter(
            x=x_range,
            y=kde(x_range),
            mode="lines",
            line=dict(color="blue"),
            name="KDE",
            showlegend=(i == 0)  # only show legend once
        ),
        row=row, col=col
    )

    # Vertical line for mean (red dashed)
    fig.add_trace(
        go.Scatter(
            x=[mean_val, mean_val],
            y=[0, max(kde(x_range)) * 1.1],
            mode="lines",
            line=dict(color="red", dash="dash"),
            name=f"Mean = {mean_val:.4f}",   # each subplot shows its own mean
            showlegend=True
        ),
        row=row, col=col
    )

# === Layout ===
fig.update_layout(
    title=dict(
        text="Residuals Distribution by Period",
        x=0.5,  # center
        xanchor="center"
    ),
    template="plotly_white",
    autosize=True,
    margin=dict(l=40, r=40, t=80, b=40)  # tighter margins
)

# Show figure responsively (fills screen)
fig.write_image("residuals_distribution.png", width=1700, height=1200, scale=2)  # 170 mm ~ 1700 px at 300 dpi

fig.show(config={"responsive": True})


We can see that they are quite different accross time! (variance of the first term is 2.5x the variance of the second term). This is mainly because some periods are calmer than others (i.e. less big surprises). The KDE plots also show how the distribution of shocks changes accross time, with some periods having fatter tails (the KDE is just a smoothed line that takes the values of the histogram, does "mini" distributions and adds then up to get a smooth line).

So we have visually checked that the variance is not constant. But let's now do it analytically, with Robert Engle's [Lagrange Multiplier method](https://en.wikipedia.org/wiki/Autoregressive_conditional_heteroskedasticity): it is a test developed by the creator of the ARCH model. It basically tells us that if we have a regressive model $y_t = X_t * \beta + \epsilon_t$ (where X_t is the vector of regressors of the trained garch model we did earlier, and beta the paremeter vector to optimize), then we formulate the following hypothesis:

- Null hypothesis (H0): No ARCH effects (constant variance)
- Alternative hypothesis : We have heteroskedasticity (i.e. changing variance)

So we first estimate the model with [Ordinary Least Squares](https://www.xlstat.com/solutions/features/ordinary-least-squares-regression-ols) (i.e. basically we try to find a line/hyperplane that is closest to the data points, i.e. fitting) so that we have a relationship between the dependent variable y and the independent variable(s) x1, x2, etc. So it is essentially a minimization problem, where we try to minimize the sum of squared shocks (errors). 

After getting this regression, we square the residuals (shocks), and then we build the following regression:

$$
\hat{\epsilon}_t^2 = \alpha_0 + \alpha_1 \hat{\epsilon}_{t-1}^2 + \alpha_2 \hat{\epsilon}_{t-2}^2 + ... + \alpha_p \hat{\epsilon}_{t-p}^2 + u_t
$$

The $\hat{\epsilon}_t^2$ are the squared residuals (shocks) we got from the first regression, and we regress them on their own lags (i.e. past values of themselves). The number of lags p is a parameter we choose!

Then we estimate again with OLS the alpha parameters, and if these params are jointly significant, i.e. if they are statistically significantly different from 0, then we reject the null hypothesis of homoskedasticity (constant variance), and we can conclude that there is heteroskedasticity (changing variance).

Once we have that reggression, we get $R^2$ = 1 - SumSquaredResiduals/SumSquaredTotal and then we compute the LM statistic: $$LM = n \cdot R^2$$, where n is the lenght of the return series. Therefore, if there are no ARCH effects, the $\hat{\epsilon}_t^2$ should not be explained by its own lags, and R^2 should be close to 0. If there are ARCH effects, then the lagged squared errors should explain a significant portion of the variance in the current squared errors, so the $R^2$ value must be higher.

$LM = n \cdot R^2$ follows a Chi-squared distribution (proved by Engle) with q degrees of freedom (where q is the number of lags we used in the regression). So we can get the p-value from this distribution, and if it is lower than 0.05 (i.e. 5% significance level), then we reject the null hypothesis of homoskedasticity (constant variance), and we can conclude that there is heteroskedasticity (changing variance).

In [ ]:
from statsmodels.stats.diagnostic import het_arch

# === 4. ARCH test for heteroskedasticity ===
print("ARCH Test for Heteroskedasticity")
print("-" * len("ARCH Test for Heteroskedasticity"))
test_stat, p_value, _, _ = statsmodels.stats.diagnostic.het_arch(res.resid)
print("ARCH test statistic:", test_stat)
print("p-value:", p_value)
if p_value < 0.05:
    print("Evidence of heteroskedasticity: the variance of the errors changes over time.")
else:
    print("No significant heteroskedasticity detected.")


ARCH Test for Heteroskedasticity
--------------------------------
ARCH test statistic: 108.75357748583721
p-value: 9.515470479583062e-19
Evidence of heteroskedasticity: the variance of the errors changes over time.


The ARCH test statistic is the LM = n * R^2 value.
The p-value is very small (below our significance level) so we reject the null hypothesis, and we can determine that there is heteroskedasticity (changing variance)!

With these plots (and the ARCH variance test), we can see that the variance is not constant accross the 4 periods (but this extends to any number of periods!). We see that the variances differ significantly, and it makes sense logically: On the first period, the 2008 crisis created more volatility (i.e. higher variance), on the second period, large shocks were rare (only the 2010 European debt crisis stands out, but in general it was a "calm" period). On the thrid period, the COVID-19 crisis increased volatility, and on the 4th period only the tariffs caused some shocks.

With this (both analytical and visual) analysis, we have explained why a model that takes into account the Heteroskedasticity of variances is important.

So, going back to the GARCH model, what about the $\alpha$ and $\beta$ parameters? They are not set manually. During the "training", they are estimated using maximum likelihood estimation ([MLE](https://www.finrgb.com/swatches/frm-part-1-garch-parameters-from-maximum-likelihood-estimation/)). The model assumes returns follow a conditional variance process:

$$
\sigma_t^2 = \omega + \alpha_1 \epsilon_{t-1}^2 + \beta_1 \sigma_{t-1}^2
$$

- $\omega$: baseline variance
- $\alpha_1$: impact of recent shocks
- $\beta_1$: persistence of past variance

MLE finds the parameter values that maximize the likelihood of the observed data, given the model. On this practice we use the arch Python package that does the parameter finding automatically. Thus, $\alpha$ and $\beta$ are learned from the data to best fit the observed returns.


In [14]:
am = arch.univariate.arch_model(rt, x=None, 
                                mean='constant', lags=None, 
                                vol='Garch', p=1, o=0, q=1, 
                                dist='normal', hold_back=None, rescale=True)

volatility_model = am.fit()
volatility_model.summary()

Iteration:      1,   Func. Count:      6,   Neg. LLF: 352826724916276.25
Iteration:      2,   Func. Count:     15,   Neg. LLF: 2828775023.9107347
Iteration:      3,   Func. Count:     22,   Neg. LLF: 9440.648006715728
Iteration:      4,   Func. Count:     29,   Neg. LLF: 6890.698359946557
Iteration:      5,   Func. Count:     35,   Neg. LLF: 6786.1328916402235
Iteration:      6,   Func. Count:     41,   Neg. LLF: 6907.090961427015
Iteration:      7,   Func. Count:     47,   Neg. LLF: 6766.282535333978
Iteration:      8,   Func. Count:     52,   Neg. LLF: 6766.279405660877
Iteration:      9,   Func. Count:     57,   Neg. LLF: 6766.277334214446
Iteration:     10,   Func. Count:     62,   Neg. LLF: 6766.277289822587
Iteration:     11,   Func. Count:     67,   Neg. LLF: 6766.277288940135
Optimization terminated successfully    (Exit mode 0)
            Current function value: 6766.277288940135
            Iterations: 11
            Function evaluations: 67
            Gradient evaluations:

<class 'statsmodels.iolib.summary.Summary'>
"""
                     Constant Mean - GARCH Model Results                      
==============================================================================
Dep. Variable:                  Close   R-squared:                       0.000
Mean Model:             Constant Mean   Adj. R-squared:                  0.000
Vol Model:                      GARCH   Log-Likelihood:               -6766.28
Distribution:                  Normal   AIC:                           13540.6
Method:            Maximum Likelihood   BIC:                           13566.6
                                        No. Observations:                 5029
Date:                Tue, Sep 23 2025   Df Residuals:                     5028
Time:                        00:43:24   Df Model:                            1
                                 Mean Model                                 
============================================================================
                 coef    std err          t      P>|t|      95.0% Conf. Int.
----------------------------------------------------------------------------
mu             0.0740  1.103e-02      6.709  1.964e-11 [5.238e-02,9.562e-02]
                              Volatility Model                              
============================================================================
                 coef    std err          t      P>|t|      95.0% Conf. Int.
----------------------------------------------------------------------------
omega          0.0281  5.533e-03      5.084  3.701e-07 [1.728e-02,3.897e-02]
alpha[1]       0.1363  1.421e-02      9.590  8.842e-22     [  0.108,  0.164]
beta[1]        0.8426  1.430e-02     58.932      0.000     [  0.815,  0.871]
============================================================================

Covariance estimator: robust
"""

### Long-term variance under GARCH(1,1)

Let's now explore a concept that I find very interesting: What happens when we want to make a very far away prediction? Do we get the mean of all the variances filtered by our model? Or is there any other alternative?

Here are the parameters of our GARCH(1,1) model:

In [71]:
volatility_model.params

mu          0.073999
omega       0.028126
alpha[1]    0.136257
beta[1]     0.842551
Name: params, dtype: float64

Let's break them down:
1. mu: This is the mean (expected return) of the time series
2. omega: This is the constant term in the variance equation (i.e. what makes volatility never go to 0)
3. alpha[1]: It measures how much yesterda's squared shock affects today's volatility
4. beta[1]: It measures how much yesterday's volatility affects today's vol. (higher beta = more persistance = more vol clustering)
5. eta: It indicates fatter tails
6. lambda: Since the model user skewed Student-t distribution, it tells us the asymmetry in returns

Therefore our GARCH(1,1) model is defined by:
$$
\sigma_t^2 = \omega + \alpha \, \epsilon_{t-1}^2 + \beta \, \sigma_{t-1}^2
$$

And with our values:

$$
\sigma_t^2 = 0.028133 \;+\; 0.136058 \,\epsilon_{t-1}^2 \;+\; 0.842692 \,\sigma_{t-1}^2
$$

*Note: if this test is rerun, yfinance will provide the *last* 20y of data, so the parameters will not be the same

Long-term variance is the level of variance that the process will converge to if you let time go to infinity (do not mistake it with the omega value!). Indeed, the long term variance is defined with:
$$
\sigma^2_{\infty} \;=\; \frac{\omega}{1 - \alpha - \beta},
$$

and since Volatility is the sqrt of the Variance:
$$
\sigma_{\infty} = +\sqrt{\frac{\omega}{1 - \alpha - \beta}}.
$$
Therefore, if we want a rough estimate of what will volatility do in a long time, we can take $\sigma_{\infty}$ as an estimate! In fact, let's compute it (and interpret the results):

In [72]:
# Retrieve Model Parameters by name
omega = volatility_model.params['omega']
alpha = volatility_model.params['alpha[1]']
beta = volatility_model.params['beta[1]']

# Retrieve conditional volatility
garch_vol = volatility_model.conditional_volatility.round(5) * np.sqrt(252)

# long-term variance under GARCH
VL = omega / (1 - alpha - beta )
print(f'Long-term variance under GARCH: {VL:.2f} / (annualized: {VL*np.sqrt(252):.2f})')

# long-term volatility under GARCH (convert from variance)
sigma_L = np.sqrt(VL) * np.sqrt(252) # already measured in percentage
print(f'Long-term volatility under GARCH: {sigma_L:.2f} %')

# sample volatility estimate
sample_sigma = rt.std() * np.sqrt(252) * 100
print(f'Sample volatility estimate: {sample_sigma:.2f} %')


Long-term variance under GARCH: 1.33 / (annualized: 21.07)
Long-term volatility under GARCH: 18.29 %
Sample volatility estimate: 19.51 %


The long term volatility tells us that our model expects returns to fluctuate about +-18.26% per year around the mean.

Sample volatility estimate: This is the actual vol we had on our dataset (annualized too)

Since the sample volatility < long-term GARCH volatility, it means that the model expects the future to be more turbulent than our test period. This tells us that GARCH is more forward looking than our sample estimate.
This result makes sense, since our dataset included fairly calm periods, and our alpha and beta values indicate high persistence of volatility

*Note: The data might be different when you run it again in the future.

### Conditional volatility

Let's now plot the returns and the conditional volatility:
This conditional volatility is the one obtained after estimating the GARCH parameters. The model goes back in time, and estimates the volatility for each day, given the past shocks and volatilities.

*Note that I have plotted them this way to see how reactive the conditional volatility is to the sudden changes in returns, not the magnitude of the spikes. In fact, they are in different scales!

In [49]:
fig = make_subplots(specs=[[{"secondary_y": True}]])

fig.add_trace(
    go.Scatter(x=garch_vol.index, y=garch_vol, name="GARCH Volatility"),
    secondary_y=False,
)

# Plot absolute value of the log returns, fixed to the bottom (y=0)
fig.add_trace(
    go.Scatter(
        x=rt.index,
        y=rt.abs(),
        name="|Log Return|",
        line=dict(color='orange'),
        fill='tozeroy',  # fill to y=0 (bottom)
        opacity=0.3
    ),
    secondary_y=True,
)

fig.add_hline(y=sigma_L, line_dash="dash", line_color="green", annotation_text="Long-run volatility estimate")
fig.add_hline(y=sample_sigma, line_dash="dash", line_color="red", annotation_text="Sample volatility")

fig.update_layout(title="GARCH(1,1) Volatility and Absolute Return")
fig.update_yaxes(title_text="Volatility", secondary_y=False)
fig.update_yaxes(title_text="|Return|", secondary_y=True)
fig.write_image("vol_vs_return_unzoom.png", width=1700, height=1200, scale=2)  # 170 mm ~ 1700 px at 300 dpi

fig.show()

We can see that our model follows the actual returns pretty well. As one could expect, it is very correlated. However, the interesting thing happens when we zoom in (preferrably somewhere with a big spike in returns). We used the COVID-19 volatility spike here:

In [51]:
# Filter data for the desired date range
start_date = "2020-02-16"
end_date = "2020-02-26"

garch_vol_zoom = garch_vol.loc[start_date:end_date]
rt_zoom = rt.loc[start_date:end_date]

fig = make_subplots(specs=[[{"secondary_y": True}]])

fig.add_trace(
    go.Scatter(x=garch_vol_zoom.index, y=garch_vol_zoom, name="GARCH Volatility"),
    secondary_y=False,
)

fig.add_trace(
    go.Scatter(
        x=rt_zoom.index,
        y=rt_zoom.abs(),
        name="|Return|",
        line=dict(color='orange'),
        fill='tozeroy',
        opacity=0.3
    ),
    secondary_y=True,
)

fig.add_hline(y=sigma_L, line_dash="dash", line_color="green", annotation_text="Long-run volatility estimate")
fig.add_hline(y=sample_sigma, line_dash="dash", line_color="red", annotation_text="Sample volatility")

fig.update_layout(title="GARCH(1,1) Volatility and Absolute Return (Feb 16 - 26, 2020)")
fig.update_yaxes(title_text="Volatility", secondary_y=False)
fig.update_yaxes(title_text="|Return|", secondary_y=True)
fig.write_image("vol_vs_ret_superzoom.png", width=1700, height=1200, scale=2)  # 170 mm ~ 1700 px at 300 dpi

fig.show()

We can see that the GARCH volatility is slightly delayed. And this makes sense, since as we explained earlier, the GARCH model is **autoregressive**, i.e. it uses its *own past data*, lagged by 1 period (days in this case).

In [52]:
# Filter data for the desired date range
start_date = "2020-02-16"
end_date = "2020-06-05"

garch_vol_zoom = garch_vol.loc[start_date:end_date]
rt_zoom = rt.loc[start_date:end_date]

fig = make_subplots(specs=[[{"secondary_y": True}]])

fig.add_trace(
    go.Scatter(x=garch_vol_zoom.index, y=garch_vol_zoom, name="GARCH Volatility"),
    secondary_y=False,
)

fig.add_trace(
    go.Scatter(
        x=rt_zoom.index,
        y=rt_zoom.abs(),
        name="|Return|",
        line=dict(color='orange'),
        fill='tozeroy',
        opacity=0.3
    ),
    secondary_y=True,
)

fig.add_hline(y=sigma_L, line_dash="dash", line_color="green", annotation_text="Long-run volatility estimate")
fig.add_hline(y=sample_sigma, line_dash="dash", line_color="red", annotation_text="Sample volatility")

fig.update_layout(title="GARCH(1,1) Volatility and Absolute Return (Feb 16-June 5, 2020)")
fig.update_yaxes(title_text="Volatility", secondary_y=False)
fig.update_yaxes(title_text="|Return|", secondary_y=True)
fig.write_image("vol_vs_ret_zoom.png", width=1700, height=1200, scale=2)  # 170 mm ~ 1700 px at 300 dpi

fig.show()

And with this other zoomed plot we can see the effects of having a high alpha+beta:
1. We see that the GARCH volatility is very persistent (it decays much slower than the returns)
2. This is why the long-run volatility estimate (GARCH) is higher than the sample volatility

In [53]:
import numpy as np
import pandas as pd
from arch import arch_model
import plotly.graph_objects as go

# === 1. Fit the GARCH model ===
am = arch_model(rt, mean='constant', vol='Garch', p=1, q=1, dist='normal', rescale=True)
res = am.fit(disp="off")

# === 2. Conditional volatility from the model ===
cond_vol = res.conditional_volatility
print("Scale factor:", res.scale)

# === 3. Realized volatility (proxy) ===
window = 5  # rolling window length
realized_vol = rt.rolling(window).std()

# === 4. Make them comparable ===
trading_days = 252
cond_vol_annual = cond_vol * np.sqrt(trading_days)
realized_vol_annual = realized_vol * np.sqrt(trading_days)

# === 5. Align indices ===
df = pd.DataFrame({
    "Conditional Vol": cond_vol_annual,
    "Realized Vol": realized_vol_annual * res.scale
}).dropna()

# === 6. Plot with Plotly ===
fig = go.Figure()

fig.add_trace(go.Scatter(
    x=df.index, y=df["Conditional Vol"],
    mode="lines",
    name="Conditional Volatility (GARCH)",
    line=dict(color="blue")
))

fig.add_trace(go.Scatter(
    x=df.index, y=df["Realized Vol"],
    mode="lines",
    name=f"Realized Volatility ({window}-day rolling)",
    line=dict(color="orange"),
    opacity=0.7
))

fig.update_layout(
    title="Conditional vs Realized Volatility (Annualized)",
    xaxis_title="Date",
    yaxis_title="Volatility (annualized)",
    legend=dict(x=0.01, y=0.99, bordercolor="Black", borderwidth=0.5),
    template="plotly_white"
)
fig.write_image("vol_vs_realized.png", width=1700, height=1200, scale=2)  # 170 mm ~ 1700 px at 300 dpi

fig.show()


Scale factor: 100.0


But why are we using a rolling window for the realized volatility? Because getting the true variance every day is too volatile, so with the rolling window, we can get a better comparison method.

## Future volatility forecasting with GARCH

We have seen how we can estimate the *past* volatility values *after* estimating its parameters. On this section however, we will focus on estimating *future* volatility, which is more useful for real life applications (where anticipating volatility levels helps hedge risks, or compute the level of exposure we have¡, like VaR modelling).

### Blind volatility prediction


Sometimes we want to estimate the volatility in N months time without estimating the market prices first (like being blind to the market). For this, we will implement volatility prediction using GARCH. We will train the model again, and for this, we will:
1. Split the data into X% training and (100-X)% testing
2. "Train" the GARCH model on the training data (i.e. get the parameters)
3. Make predictions for the test period
4. Evaluate the prediction performance with the test period's realized volatility

That is, we will predict volatility has if we had **no** information about the market (just the training data) during the test period, and then we will compare it with future test data, to see how we did

In [ ]:
# Train/Test Split and GARCH Prediction
def split_data_and_predict(returns, train_ratio=0.95):
    """
    Split data, train GARCH model, and make predictions

    Parameters:
    returns: pandas Series of returns
    train_ratio: fraction of data to use for training (default 0.9)

    Returns:
    Dictionary with train/test data, fitted model, predictions, forecasts, long-term volatility, and sample volatility
    """

    # Calculate split point
    split_point = int(len(returns) * train_ratio)

    # Split the data
    train_returns = returns.iloc[:split_point]
    test_returns = returns.iloc[split_point:]

    print(f"Training data: {len(train_returns)} observations ({train_returns.index[0]} to {train_returns.index[-1]})")
    print(f"Test data: {len(test_returns)} observations ({test_returns.index[0]} to {test_returns.index[-1]})")

    # Train GARCH model on training data
    print("\nTraining GARCH model...")
    train_model = arch.univariate.arch_model(train_returns,
                                            mean='constant', lags=None,
                                            vol='Garch', p=1, o=0, q=1,
                                            dist='normal', rescale=True)

    train_fitted = train_model.fit(disp='off')  # disp='off' to suppress output

    # Get in-sample volatility for training period
    train_volatility = train_fitted.conditional_volatility * np.sqrt(252)

    # Make predictions for the test period
    print("Making predictions for test period...")
    test_predictions = train_fitted.forecast(horizon=len(test_returns), reindex=False)
    test_vol_predictions = np.sqrt(test_predictions.variance.values[-1, :]) * np.sqrt(252)

    # Convert to pandas Series with proper dates
    test_vol_predictions = pd.Series(test_vol_predictions, index=test_returns.index)

    # Calculate actual volatility for test period using rolling window
    # For comparison, we'll use a simple approach: absolute returns scaled
    test_actual_vol = test_returns.abs() * np.sqrt(252) * 100  # Simple proxy for realized volatility

    # Calculate long-term volatility under GARCH(1,1)
    params = train_fitted.params
    omega = params['omega']
    alpha = params['alpha[1]']
    beta = params['beta[1]']
    long_term_variance = omega / (1 - alpha - beta)
    long_term_volatility = np.sqrt(long_term_variance) * np.sqrt(252)

    # Calculate sample volatility (annualized std of returns, in percent)
    sample_volatility = returns.std() * np.sqrt(252) * 100

    return {
        'train_returns': train_returns,
        'test_returns': test_returns,
        'train_fitted': train_fitted,
        'train_volatility': train_volatility,
        'test_predictions': test_vol_predictions,
        'test_actual_vol': test_actual_vol,
        'split_point': split_point,
        'long_term_volatility': long_term_volatility,
        'sample_volatility': sample_volatility
    }

# Execute the prediction
prediction_results = split_data_and_predict(rt, train_ratio=0.95)

Training data: 4777 observations (2005-09-26 00:00:00-04:00 to 2024-09-18 00:00:00-04:00)
Test data: 252 observations (2024-09-19 00:00:00-04:00 to 2025-09-22 00:00:00-04:00)

Training GARCH model...
Making predictions for test period...


And how does the prediction work? By observing the GARCH formula ($\sigma_t^2 = \omega + \alpha \epsilon_{t-1}^2 + \beta \sigma_{t-1}^2$) one would wonder: How does the model get the $\epsilon_{t-1}^2$ at each iteration without access to more information? Let's see it step by step:

On the first iteration, we have the very last $\epsilon_0^2$ and $\sigma_0^2$ of the fitted model. Therefore we have no issues getting $\sigma_1^2$:
$$
\sigma_1^2 = \omega + \alpha \, \epsilon_0^2 + \beta \, \sigma_0^2
$$

On the second iteration, we want to obtain:
$$
\sigma_2^2 = \omega + \alpha \, \epsilon_1^2 + \beta \, \sigma_1^2
$$

but we only have $\sigma_1^2$, not $\epsilon_1^2$! Instead of panicking, we use the *estimate* of $\epsilon_1^2$ i.e., we take the conditional expectation given today’s information $\mathcal{I}_0$:
$$
\mathbb{E}[\sigma_2^2 \mid \mathcal{I}_0]
= \omega + \alpha \, \mathbb{E}[\epsilon_1^2 \mid \mathcal{I}_0]
+ \beta \, \mathbb{E}[\sigma_1^2 \mid \mathcal{I}_0].
$$

(And note: $\sigma_1^2$ is already determined by the first step from $\mathcal{I}_0$, so $\mathbb{E}[\sigma_1^2 \mid \mathcal{I}_0]=\sigma_1^2$.)

And how do we get $\mathbb{E}[\epsilon_1^2 \mid \mathcal{I}_0]$? As explained before, $\epsilon_t$ is defined as:
$$
\varepsilon_t = \sigma_t z_t
$$

so $\varepsilon_t^2 = \sigma_t^2 z_t^2 \implies \mathbb{E}[\varepsilon_t^2 \mid \mathcal{I}_{t-1}] = \mathbb{E}[\sigma_t^2 z_t^2 \mid \mathcal{I}_{t-1}]$.

And if we remember, $z_t$ has mean $0$ and variance $1$. Therefore, with simple probability:
$$
\operatorname{Var}(z_t) = \mathbb{E}[z_t^2] - (\mathbb{E}[z_t])^2 \implies \mathbb{E}[z_t^2]=1.
$$

Assuming $z_t$ is independent of $\mathcal{I}_{t-1}$ (as explained before):
$$
\mathbb{E}[\varepsilon_t^2 \mid \mathcal{I}_{t-1}]
= \mathbb{E}[\sigma_t^2 \mid \mathcal{I}_{t-1}] \cdot \mathbb{E}[z_t^2]
= \mathbb{E}[\sigma_t^2 \mid \mathcal{I}_{t-1}] \cdot 1
= \mathbb{E}[\sigma_t^2 \mid \mathcal{I}_{t-1}].
$$

Apply this now to the GARCH recursion for forecasting (with $t=1$ and conditioning on $\mathcal{I}_0$):
$$
\mathbb{E}[\sigma_2^2 \mid \mathcal{I}_0]
= \omega + \alpha \, \mathbb{E}[\epsilon_1^2 \mid \mathcal{I}_0]
+ \beta \, \mathbb{E}[\sigma_1^2 \mid \mathcal{I}_0]
= \omega + \alpha \, \mathbb{E}[\sigma_1^2 \mid \mathcal{I}_0]
+ \beta \, \mathbb{E}[\sigma_1^2 \mid \mathcal{I}_0]
= \omega + (\alpha + \beta)\mathbb{E}[\sigma_1^2 \mid \mathcal{I}_0]
$$


and since, from before, we know that $\mathbb{E}[\sigma_1^2 \mid \mathcal{I}_0]= \sigma_1^2$, we get:


$$
\mathbb{E}[\sigma_2^2 \mid \mathcal{I}_0]
= \omega + (\alpha + \beta)\mathbb{E}[\sigma_1^2 \mid \mathcal{I}_0]
= \omega + (\alpha + \beta)\sigma_1^2
$$

And this pattern continues for all the next iterations. For $t \geq 1$:
$$

\mathbb{E}[\sigma_{t+1}^2 \mid \mathcal{I}_0]
= \omega + (\alpha + \beta)\, \mathbb{E}[\sigma_t^2 \mid \mathcal{I}_0]
= \omega + (\alpha + \beta)\, \sigma_t^2

$$

This is why, when forecasting volatility, the GARCH prediction converges to its long-term variance, since each new step uses the previous forecasted variance as a proxy for the (unseen) squared shock! And this is the mathematical reason why, without new information, the forecasted volatility "forgets" past shocks and drifts toward the unconditional (long-run) volatility level.

In [74]:
# Performance Evaluation
from sklearn.metrics import mean_squared_error, mean_absolute_error

def evaluate_predictions(actual, predicted, garch_longterm_vol=None):
    """
    Evaluate prediction performance and compare with GARCH long-term volatility.
    """
    # Remove any NaN values for fair comparison
    valid_indices = ~(actual.isna() | predicted.isna())
    actual_clean = actual[valid_indices]
    predicted_clean = predicted[valid_indices]
    
    if len(actual_clean) == 0:
        print("No valid data points for evaluation")
        return {}
    
    mse = mean_squared_error(actual_clean, predicted_clean)
    mae = mean_absolute_error(actual_clean, predicted_clean)
    rmse = np.sqrt(mse)
    
    # Calculate additional metrics
    mape = np.mean(np.abs((actual_clean - predicted_clean) / actual_clean)) * 100
    correlation = np.corrcoef(actual_clean, predicted_clean)[0, 1]
    
    print("Prediction Performance Metrics:")
    print("-" * 40)
    print(f"Mean Squared Error (MSE): {mse:.4f}")
    print(f"Root Mean Squared Error (RMSE): {rmse:.4f}")
    print(f"Mean Absolute Error (MAE): {mae:.4f}")
    print(f"Mean Absolute Percentage Error (MAPE): {mape:.2f}%")
    print(f"Correlation: {correlation:.4f}")
    
    # Compare GARCH long-term volatility to test period average volatility
    if garch_longterm_vol is not None:
        test_avg_vol = actual_clean.mean()
        print("\nVolatility Comparison:")
        print("-" * 40)
        print(f"GARCH Long-term Volatility: {garch_longterm_vol:.2f}")
        print(f"Test Period Average Realized Volatility: {test_avg_vol:.2f}")
    
    return {
        'mse': mse,
        'rmse': rmse,
        'mae': mae,
        'mape': mape,
        'correlation': correlation        
    }

# For test period evaluation, we'll compare with a simple realized volatility proxy
# Calculate a better realized volatility estimate using rolling standard deviation
window = 5  # 5-day rolling window
test_realized_vol = prediction_results['test_returns'].rolling(window=window).std() * np.sqrt(252) * 100
test_realized_vol = test_realized_vol.dropna()

# Align the predictions with realized volatility
aligned_predictions = prediction_results['test_predictions'].reindex(test_realized_vol.index)

# Evaluate performance and compare with GARCH long-term volatility
metrics = evaluate_predictions(
    test_realized_vol,
    aligned_predictions,
    garch_longterm_vol=prediction_results['long_term_volatility']
)

static_metrics = metrics

Prediction Performance Metrics:
----------------------------------------
Mean Squared Error (MSE): 165.3892
Root Mean Squared Error (RMSE): 12.8604
Mean Absolute Error (MAE): 8.6207
Mean Absolute Percentage Error (MAPE): 91.28%
Correlation: 0.1535

Volatility Comparison:
----------------------------------------
GARCH Long-term Volatility: 18.29
Test Period Average Realized Volatility: 14.32


Our prediction is terrible! As we can see, our volatility forecasts are off by 0.12 with respect to returns, and we miss the realized volatility by 11% (on average). Furthermore, the forecasts deviate by around 100% relative to the realized volatility, plus we also have very low correlation between the predicted and the realized volatility (i.e. the model is not tracking the short-term spikes of volatility).


However, these results are expected for a model without access to real information for about 2 years!

But the interesting thing happens when we plot the predictions, the realized volatility and their properties:

In [75]:
def plot_garch_predictions(prediction_results, sigma_L=None, avg_test_vol=None, view="all", show_train_period=True):
    """
    Plot GARCH volatility predictions and forecasts.

    Parameters
    ----------
    prediction_results : dict
        Must contain keys: 'train_volatility', 'test_predictions', 'test_returns'
    sigma_L : float, optional
    avg_test_vol : float, optional
        Long-term volatility level to plot
    view : str, optional
        "all"      → show both subplots (default)
        "full"     → only complete time series with train/test split
        "zoom"     → only zoomed test + forecast
        "forecast" → only 30-day forecast with long-term volatility
    """

    if view == "all":
        fig = make_subplots(
            rows=2, cols=1,
            subplot_titles=('Complete Time Series with Train/Test Split',
                           'Test Period Predictions and 30-Day Forecast'),
            vertical_spacing=0.1,
            row_heights=[0.6, 0.4]
        )
    else:
        fig = go.Figure()

    split_date = prediction_results['test_returns'].index[0]

    # ==== FULL SERIES VIEW ====
    if view in ["all", "full"]:
        if show_train_period:
            fig.add_trace(
                go.Scatter(
                    x=prediction_results['train_volatility'].index,
                    y=prediction_results['train_volatility'],
                    name="Training Volatility",
                    line=dict(color='blue', width=1),
                    opacity=0.7
                ),
                row=1 if view == "all" else None,
                col=1 if view == "all" else None
            )

        fig.add_trace(
            go.Scatter(
                x=prediction_results['test_predictions'].index,
                y=prediction_results['test_predictions'],
                name="Test Predictions",
                line=dict(color='red', width=2)
            ),
            row=1 if view == "all" else None,
            col=1 if view == "all" else None
        )

        fig.add_shape(
            type="line",
            x0=split_date, x1=split_date,
            y0=0, y1=1,
            yref="paper" if view == "all" else "y",
            line=dict(color="gray", width=2, dash="dot"),
            row=1 if view == "all" else None,
            col=1 if view == "all" else None
        )

        if view == "full":
            fig.update_layout(title="GARCH Volatility - Full Series")
            fig.update_yaxes(title_text="Volatility (%)")
            fig.update_xaxes(title_text="Date")

    # ==== ZOOM VIEW ====
    if view in ["all", "zoom"]:
        fig.add_trace(
            go.Scatter(
                x=prediction_results['test_predictions'].index,
                y=prediction_results['test_predictions'],
                name="Test Predictions (Zoomed)",
                line=dict(color='red', width=2),
                showlegend=(view == "zoom")
            ),
            row=2 if view == "all" else None,
            col=1 if view == "all" else None
        )

        if 'test_realized_vol' in globals():
            fig.add_trace(
                go.Scatter(
                    x=test_realized_vol.index,
                    y=test_realized_vol,
                    name="Realized Volatility",
                    line=dict(color='blue', width=1),
                    opacity=0.8
                ),
                row=2 if view == "all" else None,
                col=1 if view == "all" else None
            )

        if sigma_L is not None:
            fig.add_trace(
                go.Scatter(
                    x=[prediction_results['test_predictions'].index[0],
                       prediction_results['test_predictions'].index[-1]],
                    y=[sigma_L, sigma_L],
                    mode="lines",
                    line=dict(color="purple", width=2, dash="dot"),
                    name=f"Long-term Volatility = {sigma_L:.2f} %"
                ),
                row=2 if view == "all" else None,
                col=1 if view == "all" else None
            )
            
        if avg_test_vol is not None:
            fig.add_trace(
                go.Scatter(
                    x=[prediction_results['test_predictions'].index[0],
                       prediction_results['test_predictions'].index[-1]],
                    y=[avg_test_vol, avg_test_vol],
                    mode="lines",
                    line=dict(color="orange", width=2, dash="dot"),
                    name=f"Average Test Volatility = {avg_test_vol:.2f} %"
                ),
                row=2 if view == "all" else None,
                col=1 if view == "all" else None
            )

        if view == "zoom":
            fig.update_layout(title="GARCH Volatility - Test Period & Forecast")
            fig.update_yaxes(title_text="Volatility (%)")
            fig.update_xaxes(title_text="Date")

    # Global layout
    fig.update_layout(height=600 if view != "all" else 800, showlegend=True)
    return fig


In [58]:
# Only the full series
myplot = plot_garch_predictions(prediction_results, view="full")
myplot.write_image("garch_full_series.png", width=1700, height=1200, scale=2)  # 170 mm ~ 1700 px at 300 dpi
myplot.show()


On the first plot, we can see the training data and the prediction (without test data). Nothing surprising here.

In [78]:

# Only the zoomed test+forecast
myplot = plot_garch_predictions(prediction_results, sigma_L=prediction_results['long_term_volatility'], avg_test_vol=test_realized_vol.mean(), view="zoom")
myplot.write_image("garch_zoomed.png", width=1700, height=1200, scale=2)  # 170 mm ~ 1700 px at 300 dpi
myplot.show()


But it's on this plot where we can see some interesting things:
1. There is a 0.8% difference between the test avg. volatility and the model's long-term vol. This indicates that we trained the model on less volatile data than the one for testing.
2. Without observations of the market, the predicion converges to the long-term volatility (as expected)

In [60]:
# Additional Analysis: Forecast Statistics and Model Summary

print("GARCH Prediction and Forecast Summary")
print("=" * len("GARCH Prediction and Forecast Summary"))

# Training model summary
train_model_params = prediction_results['train_fitted'].params
print("\nTrained GARCH Model Parameters:")
print("-" * len("Trained GARCH Model Parameters:"))
for param, value in train_model_params.items():
    print(f"{param}: {value:.6f}")

# Recent vs Forecast comparison
recent_volatility = prediction_results['train_volatility'].tail(30).mean()

print(f"\nVolatility Comparison:")
print("-" * len("Volatility Comparison:"))
print(f"Recent 30-day average volatility: {recent_volatility:.2f}%")
print(f"Long-term volatility: {prediction_results['long_term_volatility']:.2f}%")


GARCH Prediction and Forecast Summary

Trained GARCH Model Parameters:
-------------------------------
mu: 0.072222
omega: 0.027085
alpha[1]: 0.135431
beta[1]: 0.844168

Volatility Comparison:
----------------------
Recent 30-day average volatility: 18.00%
Long-term volatility: 18.29%


### Daily Retrain GARCH model

On the initial model training (see Volatility Modelling with GARCH section), once we trained the model, and got the parameters, we could use it to estimate the past volatility values, by doing "filtering". We could do the same for the next days by iteratively using the GARCH formula, with the volatility obtained on the previous day and the realized shock that we would observe on the market.

While that approach is valid, it has a flaw: The parameters obtained by doing an initial "training" of the model are never again updated. On the short term, this is not a problem, but if we kept the same parameters for a long time, our model would be applying to today's forecasting a baseline volatility and persistence that was identified on the past.

In this section, I implemented a more realistic forecasting approach where:
1. We start with an initial training window (0.95% of the available data)
2. Make a 1-day ahead $\sigma_{t+1}^2$ inference (with $\sigma_{t}^2$ and $\epsilon_{t}^2$)
3. Retrain the model with the newly observed return
4. Repeat the process for each subsequent day

This approach resembles way more to the real-world forecasting where models are continuously updated with new information. However, we should note that it is more compute-intensive.

In [61]:
import warnings
warnings.filterwarnings('ignore')  # Suppress GARCH convergence warnings for cleaner output

def rolling_garch_forecast(returns, initial_window=0.90, forecast_steps=None, update_frequency=1):
    """
    Perform rolling window GARCH forecasting with model updates
    
    Parameters:
    returns: pandas Series of returns
    initial_window: fraction of data to use for initial training (default 0.8)
    forecast_steps: number of out-of-sample forecasts to make (default None → use all remaining steps)
    update_frequency: how often to re-estimate the model (1 = every day, 5 = every 5 days)
    
    Returns:
    Dictionary with rolling forecasts, actual volatility, and model parameters
    """
    
    # Calculate initial training window size
    initial_size = int(len(returns) * initial_window)
    total_obs = len(returns)
    
    # If forecast_steps not provided, use remaining observations
    if forecast_steps is None:
        forecast_steps = total_obs - initial_size
        print(f"forecast_steps set to {forecast_steps} (all remaining observations)")
    
    # Ensure we have enough data for forecasting
    if initial_size + forecast_steps > total_obs:
        forecast_steps = total_obs - initial_size
        print(f"Adjusted forecast_steps to {forecast_steps} due to data limitations")
    
    print(f"Initial training window: {initial_size} observations")
    print(f"Rolling forecasts: {forecast_steps} steps")
    print(f"Model update frequency: every {update_frequency} day(s)")
    
    # Initialize storage
    forecasts = []
    actual_returns = []
    actual_volatility = []
    forecast_dates = []
    model_params_history = []
    
    # Track when model was last updated
    last_update = -1
    current_model = None
    
    for i in range(forecast_steps):
        current_idx = initial_size + i
        
        # Determine training window end
        train_end = current_idx
        train_start = max(0, train_end - initial_size)  # Keep fixed window size
        
        # Get training data
        train_data = returns.iloc[train_start:train_end]
        
        # Update model if needed
        if current_model is None or (i - last_update) >= update_frequency:
            try:
                model = arch.univariate.arch_model(train_data, 
                                                   mean='constant', lags=None, 
                                                   vol='Garch', p=1, o=0, q=1, 
                                                   dist='normal', rescale=True)
                
                current_model = model.fit(disp='off', show_warning=False)
                last_update = i
                
                params = current_model.params.copy()
                params['update_step'] = i
                model_params_history.append(params)
                
            except Exception as e:
                print(f"Model fitting failed at step {i}: {e}")
                if current_model is None:
                    continue
        
        # Make 1-step ahead forecast
        try:
            forecast = current_model.forecast(horizon=1, reindex=False)
            vol_forecast = np.sqrt(forecast.variance.values[-1, 0]) * np.sqrt(252)
            
            forecasts.append(vol_forecast)
            actual_returns.append(returns.iloc[current_idx])
            forecast_dates.append(returns.index[current_idx])
            
            realized_vol = abs(returns.iloc[current_idx]) * np.sqrt(252) * 100
            actual_volatility.append(realized_vol)
            
        except Exception as e:
            print(f"Forecasting failed at step {i}: {e}")
            continue
        
        # if (i + 1) % 20 == 0:
        #     print(f"Completed {i + 1}/{forecast_steps} forecasts")
    
    forecasts = pd.Series(forecasts, index=forecast_dates, name='GARCH_Forecast')
    actual_volatility = pd.Series(actual_volatility, index=forecast_dates, name='Realized_Volatility')
    actual_returns = pd.Series(actual_returns, index=forecast_dates, name='Actual_Returns')
    
    print(f"\nCompleted rolling forecast with {len(forecasts)} successful predictions")
    
    return {
        'forecasts': forecasts,
        'actual_volatility': actual_volatility,
        'actual_returns': actual_returns,
        'model_params_history': model_params_history,
        'initial_window': initial_window,
        'forecast_steps': forecast_steps
    }

# Example execution
print("Starting rolling GARCH forecasting...")
rolling_results = rolling_garch_forecast(rt, initial_window=0.90, forecast_steps=None, update_frequency=1)

Starting rolling GARCH forecasting...
forecast_steps set to 503 (all remaining observations)
Initial training window: 4526 observations
Rolling forecasts: 503 steps
Model update frequency: every 1 day(s)

Completed rolling forecast with 503 successful predictions

Completed rolling forecast with 503 successful predictions


In [82]:
# Visualize Rolling Forecasts

from plotly.subplots import make_subplots
import plotly.graph_objects as go
import pandas as pd

def plot_rolling_forecasts(rolling_results, show_last_n_days=None):
    forecasts = rolling_results['forecasts'].copy()
    actual_vol = rolling_results['actual_volatility'].copy()
    actual_returns = rolling_results['actual_returns'].copy()

    # ---- 1) Build a common index and reindex everything ----
    idx = forecasts.index.union(actual_vol.index).union(actual_returns.index)
    if show_last_n_days:
        idx = idx[-show_last_n_days:]
    forecasts = forecasts.reindex(idx)
    actual_vol = actual_vol.reindex(idx)
    actual_returns = actual_returns.reindex(idx)

    # ---- 2) Subplots with a shared x-axis ----
    fig = make_subplots(
        rows=3, cols=1, shared_xaxes=True,
        subplot_titles=('Rolling GARCH Volatility Forecasts vs Realized Volatility',
                        'Forecast Errors Over Time',
                        'Actual Returns'),
        row_heights=[0.5, 0.3, 0.3]
    )

    # Top: forecasts vs actual
    fig.add_trace(go.Scatter(x=idx, y=forecasts, name="GARCH Forecasts",
                             line=dict(color='red', width=2),
                             mode='lines+markers', marker=dict(size=3)), row=1, col=1)

    fig.add_trace(go.Scatter(x=idx, y=actual_vol, name="Realized Volatility",
                             line=dict(color='blue', width=1),
                             mode='lines+markers', marker=dict(size=2), opacity=0.7), row=1, col=1)

    # Middle: errors (aligned by construction)
    forecast_errors = forecasts - actual_vol
    fig.add_trace(go.Scatter(x=idx, y=forecast_errors, name="Forecast Errors",
                             line=dict(color='green', width=1),
                             mode='lines+markers', marker=dict(size=2), showlegend=False), row=2, col=1)
    fig.add_hline(y=0, line_dash="dash", line_color="gray", row=2, col=1)

    # Bottom: returns
    fig.add_trace(go.Scatter(x=idx, y=actual_returns * 100, name="Daily Returns (%)",
                             line=dict(color='orange', width=1),
                             mode='lines', showlegend=False), row=3, col=1)

    # ---- 3) Force identical horizontal range (optional but explicit) ----
    xmin, xmax = idx.min(), idx.max()
    for r in (1, 2, 3):
        fig.update_xaxes(range=[xmin, xmax], row=r, col=1)

    # Labels & layout
    fig.update_layout(title="Rolling GARCH Forecasting Results", height=700, showlegend=True)
    fig.update_yaxes(title_text="Volatility (%)", row=1, col=1)
    fig.update_yaxes(title_text="Forecast Error", row=2, col=1)
    fig.update_yaxes(title_text="Return (%)", row=3, col=1)
    fig.update_xaxes(title_text="Date", row=3, col=1)

    return fig


# Create visualization
rolling_plot = plot_rolling_forecasts(rolling_results, show_last_n_days=150)
rolling_plot.write_image("rolling_forecasts_zoom.png", width=1700, height=1200, scale=2)  # 170 mm ~ 1700 px at 300 dpi

rolling_plot.show()

Note that here, "realized volatility" is the daily return (i.e. we are assuming the mean of returns is 0)

This plot is very similar to the initial graph we saw when exploring what the GARCH model was. Here we can also see that the the t+1 volatility is one-period lagged with respect to the returns.

But... do parameters change much over time when retraining the model every day? Let's see it:

In [ ]:
import numpy as np
import pandas as pd
import plotly.graph_objects as go
from plotly.subplots import make_subplots

# Model Parameter Evolution Analysis (with persistence, long-term volatility, and tabular stats)
def analyze_parameter_evolution(rolling_results):
    """
    Analyze how GARCH model parameters evolve over time,
    including persistence (alpha + beta) and long-term volatility.
    """
    
    params_history = rolling_results['model_params_history']
    
    if not params_history:
        print("No parameter history available")
        return
    
    # Convert to DataFrame
    params_df = pd.DataFrame(params_history)
    
    # Select key GARCH parameters
    garch_params = ['omega', 'alpha[1]', 'beta[1]']
    available_params = [p for p in garch_params if p in params_df.columns]
    
    # Add persistence if alpha and beta exist
    if 'alpha[1]' in params_df.columns and 'beta[1]' in params_df.columns:
        params_df['persistence'] = params_df['alpha[1]'] + params_df['beta[1]']
        available_params.append('persistence')
    
    # Add long-term volatility if omega, alpha, beta exist
    if all(p in params_df.columns for p in ['omega', 'alpha[1]', 'beta[1]']):
        denom = (1 - params_df['alpha[1]'] - params_df['beta[1]'])
        params_df['long_term_vol'] = np.sqrt(params_df['omega'] / denom.replace(0, np.nan))
        available_params.append('long_term_vol')
    
    if not available_params:
        print("GARCH parameters not found in history")
        return
    
    # Create parameter evolution plot
    fig = make_subplots(
        rows=len(available_params), cols=1,
        subplot_titles=[f'Parameter: {param}' for param in available_params],
        vertical_spacing=0.1
    )
    
    colors = ['blue', 'red', 'green', 'purple', 'orange']
    
    for i, param in enumerate(available_params):
        fig.add_trace(
            go.Scatter(
                x=params_df['update_step'],
                y=params_df[param],
                name=param,
                line=dict(color=colors[i % len(colors)], width=2),
                mode='lines+markers',
                marker=dict(size=4)
            ),
            row=i+1, col=1
        )
        
        # Add mean line
        param_mean = params_df[param].mean()
        fig.add_hline(
            y=param_mean, 
            line_dash="dash", 
            line_color=colors[i % len(colors)], 
            opacity=0.5,
            row=i+1, col=1
        )
        
        # Add reference line at 1 for persistence
        if param == "persistence":
            fig.add_hline(
                y=1.0, 
                line_dash="dot", 
                line_color="black", 
                opacity=0.7,
                row=i+1, col=1
            )
    
    fig.update_layout(
        title="GARCH Parameter Evolution During Rolling Forecast",
        height=400 * len(available_params),
        showlegend=True
    )
    
    # Update axis labels
    for i in range(len(available_params)):
        fig.update_yaxes(title_text="Value", row=i+1, col=1)
    fig.update_xaxes(title_text="Update Step", row=len(available_params), col=1)
    
    # --- Print parameter statistics ---
    stats = {}
    for param in available_params:
        values = params_df[param].dropna()
        if len(values) == 0:
            continue
        first_val, last_val = values.iloc[0], values.iloc[-1]
        pct_change = ((last_val - first_val) / first_val * 100) if first_val != 0 else float("nan")
        
        stats[param] = {
            "Mean": values.mean(),
            "Std": values.std(),
            "Min": values.min(),
            "Max": values.max(),
            "% Change": pct_change
        }
    
    stats_df = pd.DataFrame(stats).T  # transpose to have params as rows
    print("Parameter Evolution Statistics:")
    print("=" * 70)
    print(stats_df.round(6).to_string())
    fig.write_image("garch_parameter_evolution.png", width=1700, height=1200, scale=2)  # 170 mm ~ 1700 px at 300 dpi

    fig.show()
    
    return params_df, stats_df


# Example call:
params_evolution, stats_table = analyze_parameter_evolution(rolling_results)


Parameter Evolution Statistics:
                   Mean       Std       Min       Max  % Change
omega          0.028021  0.000890  0.026557  0.030450  6.850919
alpha[1]       0.140069  0.003949  0.135192  0.149548  8.225490
beta[1]        0.840830  0.003281  0.832814  0.845823 -1.171311
persistence    0.980899  0.001047  0.978528  0.983789  0.140830
long_term_vol  1.212466  0.039839  1.176305  1.320833  7.247437


These are very interesting results:
1. The alpha and the beta params are (more or less) inversely correlated
2. The long term volatility reacts very quickly to surprises (mostly because of the alpha component)
3. Persistence remains more or less constant throughout the training
4. As time passes, the model prioritizes progression-wise (not magnitude-wise) the shocks and less the previous volatility values


### Train and Run approach

We mentioned earlier, that after obtaining the parameters of the model, we could use the formula and the realized shocks of every passing day to calculate the conditional volatility of that day (and use it to compute the one for the next day).

The procedure is as follows:
1. Train the GARCH model on the entire training dataset (percentage of the full dataset).
2. Extract the model parameters after training.
3. Use these parameters to make predictions on the test dataset without retraining with the normal formula

With this, we want to know how such a model performs, especially compared with the previous one (that was retrained daily)

We will call this model "Recursive".

In [ ]:
import numpy as np
import pandas as pd
from arch import arch_model
import plotly.graph_objects as go

def garch_one_day_ahead(rt, split=0.9):
    """
    Run a GARCH(1,1) model on the first 'split' fraction of returns,
    then recursively compute 1-step-ahead variances for the rest.
    
    Parameters
    ----------
    rt : pd.Series
        Time series of returns
    split : float
        Fraction of sample used for initial estimation (default=0.9)
        
    Returns
    -------
    Dictionary with:
        - df: DataFrame with forecasts and realized shocks
        - res_train: fitted model on training sample
        - res_full: fitted model on full sample
        - split_idx: index that splits train/test
    """

    # Model 1: Trained daily, inferred daily
    # Model 2: Trained once on train dataset, inferred daily

    # === 1. Split sample ===
    n = len(rt)
    split_idx = int(n * split)
    train, test = rt.iloc[:split_idx], rt.iloc[split_idx:]
    
    # === 2. Fit GARCH(1,1) on training ===
    am = arch_model(train, mean="constant", vol="Garch", p=1, q=1, dist="normal", rescale=False)
    res_train = am.fit(disp="off")

    mu = res_train.params["mu"]
    omega = res_train.params["omega"]
    alpha = res_train.params["alpha[1]"]
    beta  = res_train.params["beta[1]"]
    
    # Last conditional variance from training
    sigma2_last = res_train.conditional_volatility.iloc[-1]**2
    
    # === 3. Recursive loop ===
    forecasts = []
    realized = []
    idx = []
    
    for t in range(len(test)):
        # forecast for next day (t+1)
        sigma2_next = omega + alpha * (train.iloc[-1] - mu)**2 + beta * sigma2_last
        sigma_next = np.sqrt(sigma2_next)
        
        # store
        forecasts.append(sigma_next)
        
        # realized shock of that day
        eps_realized = test.iloc[t] - mu
        realized.append(abs(eps_realized))
        
        idx.append(test.index[t])
        
        # update recursion state
        sigma2_last = sigma2_next
        train = pd.concat([train, pd.Series([test.iloc[t]], index=[test.index[t]])])
    
    # === 4. Fit full-sample model ===
    am_full = arch_model(rt, mean="constant", vol="Garch", p=1, q=1, dist="normal", rescale=False)
    res_full = am_full.fit(disp="off")
    
    # Pack results
    df = pd.DataFrame({
        "Forecast Volatility": forecasts,
        "Realized Shock": realized
    }, index=idx)
    
    return {
        "df": df,
        "res_train": res_train,
        "res_full": res_full,
        "split_idx": split_idx
    }

# === Example usage ===
results = garch_one_day_ahead(rt, split=0.5)
rolling_results = rolling_garch_forecast(rt, initial_window=0.5, forecast_steps=None, update_frequency=1)


df = results["df"]
res_train = results["res_train"]
res_full = results["res_full"]
split_idx = results["split_idx"]


# === Plot 1: Forecast vs Realized Shock ===
fig4 = go.Figure()
fig4.add_trace(go.Scatter(
    x=df.index, y=df["Forecast Volatility"]*100*np.sqrt(252),
    mode="lines", name="Train once model",
    line=dict(color="blue")
))
fig4.add_trace(go.Scatter(
    x=df.index, y=rolling_results['forecasts'],
    mode="lines", name="Daily retrain model",
    line=dict(color="orange"), opacity=0.6
))
fig4.update_layout(
    title="Daily trained (Updated model) vs Trained once (Outdated model) volatility",
    xaxis_title="Date",
    yaxis_title="Volatility% (annualized)",
    template="plotly_white",
    hovermode="x unified"
)



# Show plots
fig.write_image("retrain_trainonce.png", width=1700, height=1200, scale=2)  # 170 mm ~ 1700 px at 300 dpi

fig4.show()


c:\Users\esent\Desktop\bitpredict\venv\Lib\site-packages\arch\univariate\base.py:768: ConvergenceWarning:

The optimizer returned code 4. The message is:
Inequality constraints incompatible
See scipy.optimize.fmin_slsqp for code meaning.




forecast_steps set to 2515 (all remaining observations)
Initial training window: 2514 observations
Rolling forecasts: 2515 steps
Model update frequency: every 1 day(s)

Completed rolling forecast with 2515 successful predictions

Completed rolling forecast with 2515 successful predictions


We can see that both models are almost identical. However, we see that with time, the daily retrained model is more reactive to shocks, while having less persistence overall to volatility.

In [95]:
# Define start date
start_date = "2025-02-10"

# Slice the data
df_sub = df.loc[df.index >= start_date]
rolling_sub = rolling_results['forecasts'].loc[df.index >= start_date]

# === Plot: Train once vs Daily retrain, filtered ===
fig4b = go.Figure()
fig4b.add_trace(go.Scatter(
    x=df_sub.index, y=df_sub["Forecast Volatility"] * 100 * np.sqrt(252),
    mode="lines", name="Train once model",
    line=dict(color="blue")
))
fig4b.add_trace(go.Scatter(
    x=rolling_sub.index, y=rolling_sub,
    mode="lines", name="Daily retrain model",
    line=dict(color="orange"), opacity=0.6
))
fig4b.update_layout(
    title="Daily trained vs Train once volatility (from 2025-02-10)",
    xaxis_title="Date",
    yaxis_title="Volatility% (annualized)",
    template="plotly_white",
    hovermode="x unified"
)

fig4b.write_image("retrain_trainonce_zoom.png", width=1700, height=1200, scale=2)  # 170 mm ~ 1700 px at 300 dpi
fig4b.show()


### Comparison with VIX


The VIX is an index that measures the expected volatility of the US market over the next 30 days. Higher VIX, higher the volatility is expected to be. It is calculated from option prices of the sp500 (i.e. if investors pay more for the options, more volatility is expected).

Since our GARCH also measures volatility, let's compare them

In [ ]:
# Data & processing
ticker = yf.Ticker("^VIX")
vix = ticker.history(period = '20y', interval = "1d")
vix = vix.Close

# Clean the data by removing NaN values
garch_vol_clean = garch_vol.dropna()
vix_clean = vix.dropna()

print("Before date-only conversion:")
print(f"GARCH vol date range: {garch_vol_clean.index[0]} to {garch_vol_clean.index[-1]}")
print(f"VIX date range: {vix_clean.index[0]} to {vix_clean.index[-1]}")

# Fix timezone and time alignment by converting to date only
garch_vol_clean.index = garch_vol_clean.index.date
vix_clean.index = vix_clean.index.date

# Convert back to pandas datetime index (date only)
garch_vol_clean.index = pd.to_datetime(garch_vol_clean.index)
vix_clean.index = pd.to_datetime(vix_clean.index)

# print("After date-only conversion:")
# print(f"GARCH vol date range: {garch_vol_clean.index[0]} to {garch_vol_clean.index[-1]}")
# print(f"VIX date range: {vix_clean.index[0]} to {vix_clean.index[-1]}")

# Create DataFrame with proper alignment
val_data = pd.DataFrame({'VIX': vix_clean, 'GARCH': garch_vol_clean}).dropna()

print(f"Final combined data shape: {val_data.shape}")
print("Final data head:")
print(val_data.head())
print("...")
print(val_data.tail())

Before date-only conversion:
GARCH vol date range: 2005-09-26 00:00:00-04:00 to 2025-09-22 00:00:00-04:00
VIX date range: 2005-09-23 00:00:00-04:00 to 2025-09-22 00:00:00-04:00
Final combined data shape: (5029, 2)
Final data head:
              VIX      GARCH
2005-09-26  13.04  11.552038
2005-09-27  12.76  10.936107
2005-09-28  12.63  10.393834
2005-09-29  12.24   9.906328
2005-09-30  11.92  10.594012
...
                  VIX     GARCH
2025-09-16  16.360001  9.762187
2025-09-17  15.720000  9.423108
2025-09-18  15.700000  9.105459
2025-09-19  15.450000  9.085140
2025-09-22  16.100000  9.082917


In [106]:
# px.write_image("vix_vs_garch.png", width=1700, height=1200, scale=2)  # 170 mm ~ 1700 px at 300 dpi
px.line(val_data, line_shape='linear', title='VIX vs GARCH(1,1) Volatility').show()

They look very very similar! This means that our GARCH(1,1) model is effectively capturing the market's expectation of future volatility as represented by the VIX index.

In [107]:
print("Descriptive statistics of VIX - GARCH(1,1) volatility:", (val_data.VIX - val_data.GARCH).describe().round(2))
px.histogram(val_data.VIX - val_data.GARCH , marginal= "violin").write_image("vix_less_garch.png", width=1700, height=1200, scale=2)
px.histogram(val_data.VIX - val_data.GARCH , marginal= "violin")

Descriptive statistics of VIX - GARCH(1,1) volatility: count    5029.00
mean        2.90
std         4.65
min       -37.81
25%         1.03
50%         2.86
75%         5.10
max        28.09
dtype: float64


Looking at the distribution of the errors and the graph above, we can interpret as follows: The long left tail indicates that we sometimes overestimate the VIX significantly, while the shorter right tail suggests that our underestimations are generally smaller in magnitude, but much more frequent! If we look at the previous plot, we can see that the conditional volatility is almost always below the vix (mostly in periods of calm), but we go over the index on large spikes (which are less frequent but make our model overreact, plus the high persistence of our model makes it stay high for a while).

## Regime Classifiers

The GARCH model spits out many volatility values. However, the general public doesn't understand the difference between a squared variance of 0.04 and 0.05. They just want to know if the market is in a "high volatility", "medium" or "low volatility" period.

These abstract levels of volatility are called Regimes, and identifying which regime we were into on past (or even current) periods is called Regime Classifying. On this pratice we will use two methods: Hidden Markov Models and Markov Switching Autoregressive Model, both linked to Markov Chains.

### Hidden Markov Model


To understand HMM, we first need to understand Markov Chains.

A Markov Chain is a system that transitions from one state to another, where the probability of moving to the next state depends only on the current state, not on the sequence of events that preceded it (i.e. we only care about today to tell what we'll do tomorrow). That is, it has no memory of the past (only today matters).

So for every state, we have a probability of going to any other state (including itself). This probabilities can be stored in a matrix called the transition matrix.
$$
P = [P_{ij}], \quad \text{where } P_{ij} = P(X_{t+1} = s_j \mid X_t = s_i)
$$

And the memory-less property can be expressed as:
$$
P(X_{t+1} = s_j \mid X_t = s_i, X_{t-1}, \dots, X_0) 
= P(X_{t+1} = s_j \mid X_t = s_i)
$$

That is, the probability of transitioning to another state depends only on the current state $(X_t)$ and not on any previous states $(X_{t-1}, \dots, X_0)$. So we don't mind if we lack some farther past value.

The classical example is as follows:

Suppose we have two states:  
- $s_1 =$ Sunny  
- $s_2 =$ Rainy  
 

then the transition matrix is:

$$
P =
\begin{bmatrix}
0.8 & 0.2 \\
0.4 & 0.6
\end{bmatrix}
$$

- The rows correspond to the current state (today).  
- The columns correspond to the next state (tomorrow).  
- Each row sums to 1, because they represent probabilities of all possible next states.

So once we have the matrix, we can say that:  

- if today is Sunny ($s_1$):  
  - probability of tomorrow being Sunny = $0.8$  
  - probability of tomorrow being Rainy = $0.2$  

- if today is Rainy ($s_2$):  
  - probability of tomorrow being Sunny = $0.4$  
  - probability of tomorrow being Rainy = $0.6$  


One can start seeing what is the connection to Hidden Markov models (which is a model that assumes the memoryless property of Markov chains). On the HMM, the states are hidden (always). Continuing with the classic example, we don't know if today is sunny or rainy, but we have some observable data that depends on the state (e.g., if people are carrying umbrellas, it is more likely to be rainy).

But the "we only care about the present" filosophy is maintained. That is, the hidden states behave as a markov chain.

For a HMM, we have the following components:

1. States set: $S = \{s_1, s_2, \ldots, s_N\}$ (hidden states)  
2. Observations set: $O = \{o_1, o_2, \ldots, o_M\}$ (observable outputs)  
3. Transition probabilities: $A = [a_{ij}]$ where  
   $a_{ij} = P(X_{t+1} = s_j \mid X_t = s_i)$  
4. Emission probabilities: $B = [b_{jk}]$ where  
   $b_{jk} = P(Y_t = o_k \mid X_t = s_j)$  
5. Initial state distribution: $\pi = [\pi_i]$ where  
   $\pi_i = P(X_1 = s_i)$  

Let's ground it with the rainy/sunny example:
- States: $S = \{s_1 = \text{Sunny}, s_2 = \text{Rainy}\}$
- Observations: $O = \{o_1 = \text{No Umbrella}, o_2 = \text{Umbrella}\}$
- Transition matrix:
  $$
  A =
  \begin{bmatrix}
  0.8 & 0.2 \\
  0.4 & 0.6
  \end{bmatrix}
  $$
- Emission matrix:
  $$
    B =
  \begin{bmatrix}
  0.9 & 0.1 \\
  0.2 & 0.8
  \end{bmatrix}
  $$

The emission matrix tells us the probability of observing an umbrella or not, given the weather state. For example, if it's sunny, there's a 90% chance of seeing no umbrella, and if it's rainy, there's a 80% chance of seeing an umbrella, and a 20% chance of seeing no umbrella.

- Initial distribution: $\pi = [0.6, 0.4]$ (60% chance of starting sunny, 40% rainy)

In our use case, the umbrellas (observations) are the volatility values we gt by doing the filtering / or 1day forecast wth GARCH, and the hidden states are the volatility regimes (e.g., high volatility, medium, low volatility, ...).



So now let's apply the theory to our volatility data! We are going to use 3 hidden states (i.e. low medium and high vol).

In [108]:
def fitHMM(vol, n_states):
    # Initialize random state for reproducibility
    np.random.seed(0)

    train_vals = np.expand_dims(vol, 1)
    
    train_vals = np.reshape(train_vals,[len(vol),1])
    
    # fit Gaussian HMM to Q
    model = GaussianHMM(n_components=n_states, n_iter=1000).fit(train_vals)
     
    # classify each observation as state 0, 1 or 2
    hidden_states = model.predict(train_vals)
    post_prob = np.array(model.predict_proba(train_vals))
 
    # fit HMM parameters
    mus = np.squeeze(model.means_)
    sigmas = np.squeeze(np.sqrt(model.covars_))
    transmat = np.array(model.transmat_)
    # Print means, standard deviations, and transition matrix for inspection
    print("State means (mus):", mus)
    print("State standard deviations (sigmas):", sigmas)
    print("Transition matrix (transmat):\n", transmat)
    print("Model", model)
    
    relabeled_states = hidden_states
    return (relabeled_states, mus, sigmas, transmat, post_prob, model)

In [112]:
def plot_model(dates, vol, post_prob, export_label):
    fig = go.Figure()

    fig.add_trace(go.Scatter(x=dates, y=vol, name="GARCH", mode='lines', line_shape='hv', yaxis = 'y1'))

    fig.add_trace(go.Scatter(x=dates, y=post_prob.iloc[:,0], name = 'Pr(Low Vol Regime)', mode='lines', line_shape='hv',
                             line=dict(width=0.5, color='green'), 
                             stackgroup='two', yaxis = 'y2'))

    fig.add_trace(go.Scatter(x=dates, y=post_prob.iloc[:,1], name = 'Pr(Medium Vol Regime)', mode='lines', line_shape='hv',
                             line=dict(width=0.5, color='orange'),
                             stackgroup='two', yaxis = 'y2'))

    fig.add_trace(go.Scatter(x=dates, y=post_prob.iloc[:,2], name = 'Pr(High Vol Regime)', mode='lines', line_shape='hv',
                             line=dict(width=0.5, color='red'),
                             stackgroup='two', yaxis = 'y2'))

    # Create axis objects
    fig.update_layout(
        title = ("Volatility Regime - " + str(export_label)),

        yaxis=dict(title="Volatility"),

        yaxis2=dict(title="Posterier Probability", overlaying="y1", side="right")

    )

    # fig.write_html('Volatility Regime Classification - ' + str(export_label) + '.html') 
    fig.write_image('MSAR.png', width=1700, height=1200, scale=2)  # 170 mm ~ 1700 px at 300 dpi
    fig.show()

In [110]:
# Remove NaN values from garch_vol before fitting HMM
garch_vol_nonan = garch_vol.dropna()
hidden_states, mus, sigmas, transmat, post_prob, hmm_model = fitHMM(garch_vol_nonan, 3)
dates = garch_vol_nonan.index

hmm_data = pd.DataFrame([dates, garch_vol_nonan, hidden_states], 
                        index = ["date", "garch_vol", "Most likely state"]).T

hmm_prob = pd.DataFrame(post_prob, columns = ['state_0', 'state_1', 'state_2'])
# hmm_prob is the probability of each state at each time point
hmm_data = pd.concat([hmm_data, hmm_prob], axis=1)


hmm_data.date = pd.to_datetime(hmm_data.date)
hmm_data = hmm_data.sort_values(by="date")

hmm_data

State means (mus): [10.56188403 17.07205403 34.91428278]
State standard deviations (sigmas): [ 1.43257215  2.91549997 15.72113009]
Transition matrix (transmat):
 [[9.74158095e-01 2.45522167e-02 1.28968871e-03]
 [3.10906763e-02 9.58135226e-01 1.07740981e-02]
 [1.13255006e-54 3.45442454e-02 9.65455755e-01]]
Model GaussianHMM(n_components=3, n_iter=1000)


,date,garch_vol,Most likely state,state_0,state_1,state_2
0,2005-09-26 00:00:00-04:00,11.552038,0,1.000000,8.354838e-70,1.002092e-186
1,2005-09-27 00:00:00-04:00,10.936107,0,0.999954,4.627094e-05,1.662315e-09
2,2005-09-28 00:00:00-04:00,10.393834,0,0.999969,3.128105e-05,1.228711e-09
3,2005-09-29 00:00:00-04:00,9.906328,0,0.999977,2.314287e-05,1.946968e-09
4,2005-09-30 00:00:00-04:00,10.594012,0,0.999965,3.547101e-05,1.567149e-09
...,...,...,...,...,...,...
5024,2025-09-16 00:00:00-04:00,9.762187,0,0.999979,2.091507e-05,1.129514e-09
5025,2025-09-17 00:00:00-04:00,9.423108,0,0.999982,1.814687e-05,4.051251e-09
5026,2025-09-18 00:00:00-04:00,9.105459,0,0.999983,1.670672e-05,9.024050e-08
5027,2025-09-19 00:00:00-04:00,9.08514,0,0.999972,2.601806e-05,2.256457e-06


But how does the HMM work? Basically the model is initialized with random values for the means, variances, transition probabilities, etc... Then, it uses the [Baum-Welch algorithm](https://en.wikipedia.org/wiki/Baum%E2%80%93Welch_algorithm) (that combines two other algorithms until convergence or hitting the limit) to iteratively adjust these parameters to maximize the likelihood of the observed data (i.e. the volatilities we got from GARCH). 

The model obtains through this some n_states clusters with means and variances, and also the transition matrix.

So now let's plot it:

In [111]:
plot_model(hmm_data.date, hmm_data.garch_vol, hmm_prob, 'HMM')

It is a really nice-looking graph, but what does it mean exactly? The plot above is made up of columns (thin, but columns anyway). Each column has a colour, or several. When a column has a single color, it means that the probability of that state being the regime is 100%. If it has two colors, it means that the mode is unsure, therefore it might assign 70% to one color (one regime) and 30% to another one. The same happens when on a single column there are three colours.

NOTE: It doesn't matter which color the GARCH line touches.

This plot is the equivalent of writing that for each column (i.e. each date), there are probabilities [Pr(low vol), Pr(mid vol), Pr(high vol)]

Each column represents a row of the observation matrix, for each value of GARCH at a point in time. It is hard to see, but each column can contain the three probabilities of the states. However, the fact that it is hard to see tells us that the model switches with a lot of confidence between states (i.e. there is no period of time in which the probability of being on one of the states is not over 0.8).

We can also see that the model makes clusters of regimes (we can observe this with the 2008 financial crisis, the 2011 European debt crisis, the 2020 COVID-19 crisis, and the 2022 inflation crisis). That is, the model doesnt just indicate that there is high volatility on one day where the conditional volatility spikes, and then it returns to normality. It waits until the volatility settles to change the regime.

So we have effectively developed a model that can tell us if we are on a high, medium or low volatility regime, based on the GARCH volatility values!

### Markov Switching Autoregression Model 

However, we also see with the previous plot that the model is very sure when it changes the regime. However, in real life, this is not the case. We don't wake up one morning and say: High voaltility is over! Now we are in low volatility! Instead, we transition slowly between regimes.

To account for this property, we use the Markov Switching Autoregression Model (MSAR). This model is very similar to the HMM, but it forces the transition matrix to have high values on the diagonal (i.e. high probability of staying in the same state). This way, we force the model to be more "sticky" with the regimes. Note that the memoryless Markov property is still applied (hence the name), but we are also making past values influence the model (that's why it's called autoregressive!)

It defines properties specifically for each regime (for ex. in low volatility regime, returns follow an AR(1) with small $\sigma^2$, and on high vol, returns follow an AR(1) with a higher $\sigma^2$ and possibly different mean reversion)

In [ ]:
# Fit the model
mod_hamilton = sm.tsa.MarkovAutoregression((rt-rt.mean()).dropna(), k_regimes=3, order = 1, trend="n", switching_ar = False, switching_variance = True)
    
res_hamilton = mod_hamilton.fit()
res_hamilton.summary()

<class 'statsmodels.iolib.summary.Summary'>
"""
                         Markov Switching Model Results                         
================================================================================
Dep. Variable:                    Close   No. Observations:                 5028
Model:             MarkovAutoregression   Log Likelihood               16446.801
Date:                  Tue, 23 Sep 2025   AIC                         -32873.601
Time:                          00:45:54   BIC                         -32808.374
Sample:                               0   HQIC                        -32850.746
                                 - 5028                                         
Covariance Type:                 approx                                         
                             Regime 0 parameters                              
==============================================================================
                 coef    std err          z      P>|z|      [0.025      0.975]
------------------------------------------------------------------------------
sigma2      2.774e-05   8.79e-07     31.563      0.000     2.6e-05    2.95e-05
                             Regime 1 parameters                              
==============================================================================
                 coef    std err          z      P>|z|      [0.025      0.975]
------------------------------------------------------------------------------
sigma2         0.0001        nan        nan        nan         nan         nan
                             Regime 2 parameters                              
==============================================================================
                 coef    std err          z      P>|z|      [0.025      0.975]
------------------------------------------------------------------------------
sigma2         0.0010        nan        nan        nan         nan         nan
                           Non-switching parameters                           
==============================================================================
                 coef    std err          z      P>|z|      [0.025      0.975]
------------------------------------------------------------------------------
ar.L1         -0.0617      0.014     -4.260      0.000      -0.090      -0.033
                         Regime transition parameters                         
==============================================================================
                 coef    std err          z      P>|z|      [0.025      0.975]
------------------------------------------------------------------------------
p[0->0]        0.9707      0.005    202.181      0.000       0.961       0.980
p[1->0]        0.0333      0.005      6.411      0.000       0.023       0.043
p[2->0]     2.115e-06        nan        nan        nan         nan         nan
p[0->1]        0.0289      0.005      5.882      0.000       0.019       0.038
p[1->1]        0.9606      0.006    167.531      0.000       0.949       0.972
p[2->1]        0.0383      0.013      3.019      0.003       0.013       0.063
==============================================================================

Warnings:
[1] Covariance matrix calculated using numerical (complex-step) differentiation.
"""

In [113]:
post_prob = res_hamilton.smoothed_marginal_probabilities
post_prob = pd.DataFrame(post_prob)

plot_model(dates, garch_vol, post_prob, 'MSAR garch_vol')

And exactly as expected, here we can see that the model sticks more to the states (i.e. there are more "curves", which mean that the probabilities of being in a state change more slowly).

### Comparison

A useful method for comparing how well each model fits the data is the log-likelihood. The log-likelihood measures how probable the observed data is, given the model parameters. A higher log-likelihood indicates a better fit to the data.

In [114]:
hmm_log_prob = hmm_model.score(np.expand_dims(garch_vol.dropna(),1))
print('log-likelihood of HMM:', hmm_log_prob)
print('Transition Matrix of HMM:')
print(hmm_model.transmat_)

log-likelihood of HMM: -12596.045936794613
Transition Matrix of HMM:
[[9.74158095e-01 2.45522167e-02 1.28968871e-03]
 [3.10906763e-02 9.58135226e-01 1.07740981e-02]
 [1.13255006e-54 3.45442454e-02 9.65455755e-01]]


In [115]:
msar_log_prob = mod_hamilton.loglike(res_hamilton.params)
trans_matrix = mod_hamilton.regime_transition_matrix(res_hamilton.params)
print('Log-likelihood of MSAR:', msar_log_prob)
print('Transition Matrix of MSAR:')
print(trans_matrix)

Log-likelihood of MSAR: 16446.800669230575
Transition Matrix of MSAR:
[[[9.70666336e-01]
  [3.33071545e-02]
  [2.11475002e-06]]

 [[2.88565606e-02]
  [9.60615885e-01]
  [3.83404736e-02]]

 [[4.77103580e-04]
  [6.07696087e-03]
  [9.61657412e-01]]]


We see that both models say that states are very persistent (over 95% of staying). The higher log-likelihood value of the MSAR indicates that it is a better fit for the input data (i.e. the GARCH volatilities). This is expected, since the MSAR model accounts for the persistence of volatility regimes, which is a known characteristic of financial markets.

However, we can see that the MSAR model is not necessarily more "sticky" than the HMM. In fact, the HMM has a higher probability of staying in the low volatility regime and the high volatility regime. The MSAR model has a higher probability of staying in the medium volatility regime. This means that the MSAR model is more likely to switch between low and high volatility regimes, while the HMM model is more likely to stay in the same regime.

## Conclusion

This practice has been much more than just an exercise in applying volatility models. It has been a way for me to truly understand how modern methods like GARCH and regime-switching models work in practice, and what they can (and cannot) tell us about financial markets.

Paradoxically, I intended to make it not very rigorous (I wanted to focus on understanding the topic, rather than proving everything around it), but I ended up discovering many statistical tests that help check the assumptions behind these models and to choose between them.

The real value, though, has gone beyond the models themselves. Working through the material has made me more aware of the assumptions hidden in the simplest things we often take for granted, like how we define returns, or what we assume about prices and distributions. It has also shown me how research in finance differs from pure mathematics: while math often focuses on very intrincate theorems, finance requires constant confrontation with messy, imperfect data.

Along the way, I’ve revisited key probability, explored hypothesis testing, and discovered new areas of quantitative finance. I’ve also seen how GARCH models can be useful in real applications, while at the same time realizing their limitations. Most importantly, I’ve learned the value of persistence: to keep digging until I find an explanation that makes sense to me, even when the literature feels too dense or overly technical.

In the end, this notebook has been both a learning tool and a personal exploration by helping me connect theory with practice, and motivating me to keep asking questions that lead to deeper understanding.

Disclaimer on the use of AI:

I have used AI assistants (ChatGPT) for coding and formatting the formulas on the text. Even though the main purpose of this practice is to understand the concepts, I ended up becoming very familiar with the code.

## References  
**By section:**  
1. Return distributional Assumptions  
> ARCH Model https://www.fsb.miamioh.edu/lij14/672_2014_s5.pdf  
> Heterogeneous Auroregressive Mean Model https://arch.readthedocs.io/en/latest/univariate/generated/arch.univariate.HARX.html#arch.univariate.HARX  
> Garch Forecasting Performance under Different Distribution Assumptions http://www-stat.wharton.upenn.edu/~steele/Courses/434/434Context/GARCH/Willhelmesson06.pdf  

2. Volatility Modelling
> Predicting volatility with heterogeneous autoregressive models https://www.sr-sv.com/predicting-volatility-with-heterogeneous-autoregressive-models/   


3. Hidden Markov Model
> Practical Time Series Analysis - code repo https://github.com/PracticalTimeSeriesAnalysis/BookRepo      
> HMMLearn https://hmmlearn.readthedocs.io/en/latest/
> Quantstrat HMM https://www.quantstart.com/articles/market-regime-detection-using-hidden-markov-models-in-qstrader/

4. Markov Switching Autoregressive Model
> ECB Volatility Regime https://www.ecb.europa.eu/pub/financial-stability/fsr/focus/2018/pdf/ecb~bcaaae16c3.fsrbox201805_03.pdf  
> Autoregressive conditional heteroskedasticity and changes in regime https://www.sciencedirect.com/science/article/abs/pii/0304407694900671    
> Markov-Switching - Kim, Nelson, and Startz (1998) Three-state Variance Switching http://www.chadfulton.com/topics/mar_kim_nelson_startz.html   
> Statsmodels Variance Switching Model https://www.statsmodels.org/dev/examples/notebooks/generated/markov_autoregression.html#Kim,-Nelson,-and-Startz-(1998)-Three-state-Variance-Switching  
> Statsmodels Markov Regression https://www.statsmodels.org/devel/examples/notebooks/generated/markov_regression.html   
